# ALS reconstruction and KRG-alsDenoise, v3.3

Reconstruction and self-supervised denoising of 180° ALS micro-CT scans.
The workflow creates full/even/odd reconstruction stacks, estimates the
center of rotation with TomoPy Vo, and manages reconstruction queues,
training, inference and loss-history plots.

This is a historical notebook. The maintained entry point is
[v3.4](Noise2Inverse360_HPC_v3.4.ipynb). Earlier notebook code is retained
for existing runs; installation instructions and helper code differ by version.


## Method and input requirements

Noise2Inverse360 trains a denoising model without a clean reference image. For each slice, it compares two reconstructions made from different measured angles: even-indexed projections and odd-indexed projections. The model learns features that agree between the two views. A third reconstruction from all projections supplies the image that will be denoised after training. The two training views share flats and darks, so their noise is not perfectly independent; inspect the result for removed features and remaining artifacts.

This notebook performs reconstruction and denoising. The ALS `microct` code performs the reconstruction; `KRG-alsDenoise` supplies the Noise2Inverse model training and inference commands. Reconstruction settings, especially center of rotation, must match across the full, even, and odd stacks. The notebook saves one plan and applies it to all three.

| Term | Meaning here |
|---|---|
| Projection | A raw 2D X-ray image acquired at one rotation angle. |
| Flat / dark | Calibration images acquired without the sample / without X-ray illumination. |
| Sinogram row | One detector row followed through all angles; it reconstructs one CT slice. |
| Center of rotation | Detector-center offset in pixels: `native center - detector width / 2`. Zero is the midpoint; negative offsets are valid. A wrong value can produce doubled edges. |
| Full / even / odd stack | TIFF slices reconstructed from all / even-indexed / odd-indexed projections. |
| 2.5D model | A 2D network that reads several neighboring slices as channels. |
| 3D model | A 3D network that reads and predicts cubic patches of the volume. |
| HDF5 virtual view | A small HDF5 file pointing at selected projections in the original file; it does not duplicate the raw images. |

Required input: one 180° parallel-beam HDF5 scan containing `/exchange/data` in `[angle, row, column]` order, `/exchange/theta` in degrees, and `/exchange/data_white` and `/exchange/data_dark`. The ALS path also needs detector dimensions, pixel size, angle count, sample distance, energy, angular range, and other standard ALS metadata. Cell 11 checks image layout and ALS metadata; cell 12 rechecks metadata before reconstruction. A folder of only reconstructed TIFFs cannot supply the independent even/odd training views; obtain the raw projections or separately reconstructed views before using Noise2Inverse360.

The detector width must be even. Flats and darks must be nonempty `[frame, row, column]` arrays matching the projections. Angles must increase monotonically and span approximately 180°. Metadata values used below must be finite, nonempty 1D arrays, not HDF5 scalar datasets.

| Required ALS metadata path | Meaning used by this notebook |
|---|---|
| `/measurement/instrument/detector/dimension_x`, `dimension_y` | Column and row counts, matching `/exchange/data`. |
| `/measurement/instrument/detector/pixel_size` | First entry: detector pixel size in **mm**; must be positive. |
| `/process/acquisition/rotation/num_angles`, `range` | Projection count and angular span in **degrees**, consistent with the image data and theta. |
| `/measurement/instrument/camera_motor_stack/setup/camera_distance` | Optional for this no-phase workflow. If present as a finite second entry, it is reported in mm; otherwise distance is recorded as unknown. A private metadata adapter avoids upstream indexing entry 1 when it is absent. |
| `/measurement/instrument/monochromator/energy` | First entry in **eV**. Required by the reader even though this notebook does not perform phase retrieval. |

A finite negative energy value can pass the reader but is not a valid physical energy; it does not enable phase retrieval. Obtain missing or malformed metadata from the beamline rather than substituting guessed values. The pixel-size conversion used for reconstruction is mm → cm, giving attenuation values in **1/cm**.

## First-time NERSC setup

This guide starts with a NERSC account, access to the raw scan, and permission to
clone `remyferu/KRG-alsDenoise`. No earlier notebook or installed CT environment
is required. Commands marked terminal are Bash commands on NERSC, not Python
notebook cells. Run them on NERSC, not on a local Windows host. A kernel is the
Python process that executes notebook cells; selecting it is separate from
selecting the compute node on which Jupyter runs.

### 1. Environment roles

| Environment | What runs there | Main software | How it is selected |
|---|---|---|---|
| `als832` | Notebook cells, projection processing, COR and reconstruction | ALS `microct` source, TomoPy, ASTRA, dxchange, NumPy, widgets | Select **ALS CT + N2I notebook** as the Jupyter kernel |
| `n2i` | Denoising training and inference | This repository, PyTorch with CUDA | Notebook launches its absolute `denoise` executable as a subprocess |

Create both environments once. Keep the notebook in the ALS kernel during the
whole workflow. Training runs in an n2i subprocess.
The source checkouts contain code, while the Conda prefixes contain Python and
installed packages. Cloning a repository alone does not install its dependencies.

This notebook reconstructs through ALS/TomoPy/ASTRA; no Tomocupy environment is
required. Cell 21 runs `denoise prepare` to create a training YAML. That command
may print Tomocupy reconstruction examples inherited from the original CLI;
ignore those examples here because cells 9–11 already produce and check the paired
stacks using ALS.

An existing ALS-provided `microCT` environment can serve as
the kernel after verifying the imports below. The
[ALS project](https://github.com/als-computing/microct#readme) describes its
beamline-specific service. The private installation below is an alternative for
standard Perlmutter Jupyter sessions. Their filesystem mounts may differ.

### 2. Create directories and download the source (login-node terminal)

Log in to NERSC through SSH, or use a login-node Jupyter terminal for lightweight
installation work. Use a compute allocation for expensive compilation or data
processing. These paths match the notebook's defaults. The Conda module follows
the [NERSC custom-environment procedure](https://docs.nersc.gov/development/languages/python/nersc-python/#option-2-custom-conda-environment):

```bash
module load conda
CT_SOURCE_DIR="$HOME/microct-n2i-source"
CT_SOFTWARE_DIR="$HOME/microct-n2i-envs"
mkdir -p "$CT_SOURCE_DIR" "$CT_SOFTWARE_DIR"
cd "$CT_SOURCE_DIR"
git clone https://github.com/als-computing/microct.git microct
git -C "$CT_SOURCE_DIR/microct" checkout e4116984003294af8dabba9d32db3fccb5ef2253
git clone https://github.com/remyferu/KRG-alsDenoise.git KRG-alsDenoise
```

The private KRG repository requires GitHub HTTPS authentication or
a configured SSH key with `git@github.com:remyferu/KRG-alsDenoise.git` as the
clone URL. If a directory already exists, inspect it and reuse the checkout;
do not rerun `git clone` into it or replace it. The ALS commit above matches the
backend reviewed for this notebook.

### 3. Install the ALS reconstruction environment (login-node terminal)

For a fresh private installation, create `als832` from the pinned ALS source's
environment file, then add the notebook's supporting packages:

```bash
module load conda
CT_SOURCE_DIR="$HOME/microct-n2i-source"
CT_SOFTWARE_DIR="$HOME/microct-n2i-envs"
conda env create --prefix "$CT_SOFTWARE_DIR/als832" \
  -f "$CT_SOURCE_DIR/microct/environment.yml"
conda activate "$CT_SOFTWARE_DIR/als832"
conda install -c conda-forge h5py tifffile pyyaml pillow numexpr ipywidgets ipykernel
python -c 'import tomopy, dxchange, astra, h5py, tifffile, yaml, PIL, ipywidgets; print("ALS imports OK")'
python -m ipykernel install --user --name als832-n2i \
  --display-name "ALS CT + N2I notebook"
```

The [pinned ALS environment](https://github.com/als-computing/microct/blob/e4116984003294af8dabba9d32db3fccb5ef2253/environment.yml)
uses Python 3.8 and older Matplotlib/Jupyter pins. This is a source-based setup
recipe, not a freshly tested NERSC package lock. If Conda cannot solve it, keep an
existing working ALS environment and resolve the solver error before proceeding;
do not install TomoPy into n2i as a substitute. `ipywidgets` is needed by the ALS
backend itself. Zoom displays do not require an interactive Matplotlib backend.

In JupyterLab choose **Kernel → Change Kernel → ALS CT + N2I notebook**. Reload
JupyterLab if the kernel is missing. Changes to an already running kernel's
environment require a kernel restart. The first code cell prints `sys.executable`;
for this installation it should end in `microct-n2i-envs/als832/bin/python`.
See the [NERSC custom-kernel procedure](https://docs.nersc.gov/services/jupyter/how-to-guides/#how-to-use-a-conda-environment-as-a-python-kernel).

### 4. Install the denoising environment (login-node terminal)

For a new n2i prefix, install a CUDA-enabled PyTorch build and this package with
its dependencies. The example uses an official PyTorch 2.7.1 / CUDA 11.8 wheel;
the notebook requires PyTorch >= 2.0. Use a site-compatible build when necessary.
[PyTorch wheel choices](https://pytorch.org/get-started/previous-versions/#v271)
and [NERSC guidance](https://docs.nersc.gov/machinelearning/pytorch/) explain the
custom-environment option.

```bash
module load conda
CT_SOURCE_DIR="$HOME/microct-n2i-source"
CT_SOFTWARE_DIR="$HOME/microct-n2i-envs"
conda create --prefix "$CT_SOFTWARE_DIR/n2i" python=3.11 pip
conda activate "$CT_SOFTWARE_DIR/n2i"
python -m pip install 'numpy<2.0'
python -m pip install torch==2.7.1 --index-url https://download.pytorch.org/whl/cu118
python -m pip install -e "$CT_SOURCE_DIR/KRG-alsDenoise"
python -m pip check
MPLBACKEND=Agg PYTHONNOUSERSITE=1 "$CT_SOFTWARE_DIR/n2i/bin/denoise" train -h
```

Help must contain `--architecture` and its `nafnet25d` choice. There is no need to
clone NAFNet separately or install BasicSR. For an **existing working n2i prefix**,
skip `conda create` and the dependency installations; activate that prefix and run
`python -m pip install --no-deps -e "$HOME/microct-n2i-source/KRG-alsDenoise"`,
then check imports and CLI help. `--no-deps` is only for an environment with the
dependencies already installed. The repository's original
`envs/denoise_environment.yml` is a machine-specific upstream export, not the
portable first-time recipe for this notebook.

### 5. Start Jupyter on a compute allocation

Open [NERSC JupyterHub](https://jupyter.nersc.gov/), start a **single-node GPU**
compute server and open
`microct-n2i-source/KRG-alsDenoise/notebooks/Noise2Inverse360_HPC_v3.3.ipynb`.
For the full-volume example, choose **Exclusive GPU Node**, or a Configurable job
with sufficient host RAM and time. NERSC documents four GPUs and full node memory
for the exclusive GPU configuration; a shared GPU server has one GPU and a
quarter of the node's host resources. [Allocation options](https://docs.nersc.gov/services/jupyter/reference/).

Select **ALS CT + N2I notebook** as the kernel. Activating Conda or starting an
allocation in another terminal does not relocate or change the running kernel.
The workflow uses one node: training uses all assigned GPUs, inference uses one.
In a terminal opened **inside this compute Jupyter session**, check:

```bash
hostname
echo "Slurm job: ${SLURM_JOB_ID:-unset}"
echo "GPU assignment: ${CUDA_VISIBLE_DEVICES:-unset}"
nvidia-smi
"$HOME/microct-n2i-envs/n2i/bin/python" -c 'import torch; print(torch.__version__, torch.version.cuda); assert torch.cuda.is_available(); print("Visible GPUs:", torch.cuda.device_count())'
```

This checks CUDA availability; it does not train a model. Keep the scheduler's
`CUDA_VISIBLE_DEVICES` assignment. If it is unset, resolve the allocation/kernel
setup before training instead of guessing identifiers from `nvidia-smi`. If either
environment needs site CUDA modules, apply them through a kernel/helper wrapper
following [NERSC's helper instructions](https://docs.nersc.gov/services/jupyter/how-to-guides/#how-to-customize-a-kernel-with-a-helper-shell-script).
The absolute executable path alone cannot repair a missing native runtime.

### 6. Stage the scan and edit cell 10

The notebook does not download or copy scan data. Place the raw HDF5 scan in a
directory visible from the compute kernel. On standard Perlmutter, an example is
`$PSCRATCH/raw_microct/sample.h5`. Set the source filename and verify that
the file meets the projection/flat/dark/metadata requirements in **Method and input requirements**.
Reconstructed TIFFs alone are insufficient for the notebook's projection splitting.

| Cell 10 input | What to enter |
|---|---|
| `INPUT_SUBFOLDER` / `INPUT_ROOT` | Folder containing raw `.h5`/`.hdf5` files; replace the example folder or use an explicit absolute `Path` |
| `PREFERRED_FILE` | Raw scan filename; confirm the selected scan in cell 11 |
| `RUN_LABEL` | A name for this scan's particular COR/settings/model combination; use a new label when those change |
| `ALS_MICROCT_DIR` | Checkout containing `backend/ALS_recon_helper.py`, normally `Path.home() / 'microct-n2i-source' / 'microct'` |
| `DENOISE_CMD` | `[str(Path.home() / 'microct-n2i-envs' / 'n2i' / 'bin' / 'denoise')]` |
| `OUTPUT_ROOT_OVERRIDE` | Leave `None` to use `$PSCRATCH`/`$SCRATCH`, or enter an absolute output scratch directory |

In Python, `Path.home()` expands the account home directory; `Path('$HOME/...')` does not
expand Bash variables. `/alsdata` and `/alsuser/pscratch` are service-specific
paths, not assumed mounts on standard Perlmutter. Stage to a stable input path:
moving the raw file later breaks the saved plan and virtual projection views.

### 7. Directory layout

Keep code, environments and notebooks in home (subject to quota); use scratch for
large input copies, TIFFs and active model outputs. Standard output layout:

```text
$HOME/microct-n2i-source/microct/backend/ALS_recon_helper.py
$HOME/microct-n2i-source/KRG-alsDenoise/notebooks/Noise2Inverse360_HPC_v3.3.ipynb
$HOME/microct-n2i-envs/als832/         # notebook Python
$HOME/microct-n2i-envs/n2i/           # denoising Python
$PSCRATCH/noise2inverse360/<scan>/<RUN_LABEL>/
  run_plan.json, scan_metadata.json
  projection_views/even.h5, odd.h5
  <scan>_rec/, <scan>_rec_0/, <scan>_rec_1/
  <scan>_rec_config.yaml
  TrainOutput/results/reference_metrics.csv
  TrainOutput/{resume,best_val_model,best_lcl_model,best_edge_model}.pth
  <scan>_denoised_slices_nafnet25d/
  <scan>_denoised_volume_2.5d_nafnet25d/
  logs/, training_history/
$PSCRATCH/noise2inverse360/reconstruction_queue_v3.json
```

The selected output root can differ from this example. Full/even/odd float32
stacks for 624 × 2560 × 2560 occupy about 46 GiB on disk. Training with four
processes holds about 122 GiB of paired input arrays in host RAM before overhead.
GPU VRAM is separate from host RAM. Read the volume section for its additional
memory needs. Scratch is working storage; preserve accepted data and small run
records in durable storage before cleanup.

### 8. Update an existing installation

After stopping jobs that use this checkout, update it in a terminal:

```bash
cd "$HOME/microct-n2i-source/KRG-alsDenoise"
git status --short
git pull --ff-only
```

Preserve local notebook edits before pulling. Editable installation reads code
from this checkout, so routine code updates need no package reinstall; changed
dependencies or entry points do. Reopen the updated notebook and restart/load its
helper cells. Do not update the pinned ALS checkout during an active run: changing
backend files invalidates its saved source identity.

### Setup troubleshooting

| Symptom | Next step |
|---|---|
| `ModuleNotFoundError: tomopy`, `astra` or `dxchange` in a cell | Check the printed kernel Python; select the ALS kernel and verify ALS imports |
| `ModuleNotFoundError: torch` from `denoise` | Check `DENOISE_CMD` and the n2i environment; PyTorch belongs there |
| CLI help lacks `--architecture` | Update/install KRG-alsDenoise in n2i; an older Noise2Inverse executable is being used |
| Invalid `matplotlib_inline` backend in the CLI | Confirm the subprocess uses `MPLBACKEND=Agg`; the notebook's launcher sets it |
| NumExpr thread-count error | Restart the kernel and run cell 1 first. If NumExpr is preloaded, set `NUMEXPR_MAX_THREADS` in the kernel launcher to at least the host's detected logical CPU count before Python starts |
| Kernel remains missing | Reload JupyterLab after registration; check the kernelspec points to an existing ALS Python |
| No accessible scratch or scan folder | Set cell 10's input and scratch paths to locations visible on the actual compute node |

If NumExpr loads before cell 1, inspect the host CPU count with
`getconf _NPROCESSORS_ONLN` in the compute-session terminal. In JupyterLab's text
editor open `$HOME/.local/share/jupyter/kernels/als832-n2i/kernel.json` and add an
`env` mapping at the same level as `argv` (keep the existing interpreter path):

```json
"env": {
  "NUMEXPR_MAX_THREADS": "256",
  "NUMEXPR_NUM_THREADS": "8"
}
```

`256` is an example: use a maximum at least as large as that host's detected
logical CPU count. Keep valid JSON commas between the existing fields, then
restart the kernel. Cell 1 reduces the initial active count to the available CPU
limits. [NERSC kernel environment instructions](https://docs.nersc.gov/services/jupyter/how-to-guides/#how-to-set-environment-variables-for-a-python-kernel).

Proceed with cells **1–9** to load the notebook helpers, then follow **Workflow and
restart**. Training is enabled by setting `RUN_TRAINING=True` in cell 22. This setup has not been verified through a fresh NERSC installation
and reconstruction. [Denoising details](../docs/NAFNET_CT.md) cover architecture and loss settings.


## Workflow and restart

After first-time setup, run individual cells in the order below. **Do not use Run
All for a first scan:** settings and image checks require review. Code cell
numbers appear at the beginning of each cell.

| Cells | Action and expected result |
|---|---|
| 1–9 | Run cell 1 first, then every helper cell; they define functions and configure threads without reading a scan |
| 10–11 | Edit paths, select a raw scan and inspect its dimensions/metadata |
| 12 | Choose reconstruction settings and verify the ALS backend |
| 13 | Optional first/opposite projection alignment check |
| 14 | Estimate center of rotation (COR); a **DONE** message means `run_plan.json` is saved |
| 15–16 | Optional manual COR review; needed when cell 14 reports **REVIEW REQUIRED**. Apply the selection, then save it |
| 17–18 | Add the saved plan to the queue, then reconstruct full/even/odd stacks |
| 19 | Inspect matching reconstructed rows in all three stacks |
| 20–21 | Choose 2.5D/3D and configure training; this writes YAML but does not train |
| 22 | Train/resume only after setting `RUN_TRAINING=True`; logs and history are saved automatically |
| 23 | Optional plot refresh or plotting a separate existing log |
| 24–25 | Select a checkpoint, inspect denoising, then run volume inference. 3D has no separate slice command |
| 26–27 | Optional queue inspection and explicit repair |
| 28–29 | Optional generated-output cleanup: preview first, then opt in explicitly |

The **plan** freezes source identity, COR and reconstruction settings. The **queue**
is a list of plans waiting to reconstruct; enqueueing alone does not run them.
**Full** uses all projections; **even/odd** use alternating projection indices.
**Training** learns weights; **inference** applies a saved checkpoint. `DONE` confirms
the named action, not the scientific accuracy of its result.

**Resume after a kernel restart:** select the ALS kernel, run cells 1–9, and rerun
10–12 with the same scan, `RUN_LABEL` and reconstruction settings. Run cell 14 to
restore the saved COR/plan without recomputing it. Cells 17–18 recover interrupted
reconstruction, validating and skipping completed stages. Run 19–21 again, select
the original architecture/mode, then enable 22 to resume from `resume.pth`.
If `TrainOutput` exists without that checkpoint, preserve it and use a new run
label for a fresh run. Cell 23 can plot an existing log without training.

**Change settings:** use a new run label for new COR, reconstruction, mode,
architecture or slice count. Source/backend changes invalidate saved reconstruction
identity. Switching scans invalidates older widget controls; recreate them by
rerunning the relevant cells. A restart clears Python variables but preserves files.

**Interrupt:** use Jupyter's interrupt control, allow the cell to finish its cleanup,
then inspect its log before restarting. Notebook-launched denoising processes are
terminated on an interrupt; external jobs must be stopped separately. Do not delete
models or logs merely to resume. Full-volume inference has no resume checkpoint;
preserve partial output before retrying into a new directory/run.

**NumExpr thread errors:** restart and run cell 1 before scientific imports. Its
initial active thread count is at most eight; this is not a global eight-core limit
on reconstruction. The import-time maximum accommodates TomoPy's host CPU-count
requests. If NumExpr was preloaded during kernel startup, follow the setup
troubleshooting steps before restarting again.


In [ ]:
# 1. Load libraries and configure threads

from pathlib import Path
from collections import deque
import uuid
import socket
import stat
from datetime import datetime, timezone
import contextlib
import copy
from PIL import Image
import io
import base64
import csv
import hashlib
import html
import importlib.util
import inspect
import json
import os
import re
import shlex
import shutil
import signal
import subprocess
import sys
import time

def configure_reconstruction_threads():
    """Configure NumExpr before scientific imports; cover TomoPy's CPU-count requests."""
    detected = max(1, os.cpu_count() or 1)
    available = detected
    if hasattr(os, 'sched_getaffinity'):
        try:
            available = min(available, max(1, len(os.sched_getaffinity(0))))
        except OSError:
            pass
    for name in ('SLURM_CPUS_PER_TASK', 'NUMEXPR_NUM_THREADS'):
        value = os.environ.get(name, '')
        if value.isdecimal() and int(value) > 0:
            available = min(available, int(value))
    # TomoPy defaults some NumExpr operations to the host CPU count, even
    # when fewer CPUs are assigned. Its requests must fit the import-time pool.
    maximum = max(64, detected)
    active = min(8, available)
    os.environ['NUMEXPR_MAX_THREADS'] = str(maximum)
    os.environ['NUMEXPR_NUM_THREADS'] = str(active)
    loaded = sys.modules.get('numexpr')
    if loaded is not None:
        if loaded.MAX_THREADS < detected:
            raise RuntimeError(
                f'NumExpr was already imported with a {loaded.MAX_THREADS}-thread pool, '
                f'but TomoPy can request {detected}. Restart the kernel and run this '
                'imports cell first. If NumExpr loads during kernel startup, set '
                f'NUMEXPR_MAX_THREADS={maximum} in the kernel launcher before Python starts.')
        active = min(active, loaded.MAX_THREADS)
        if loaded.set_num_threads(active) < 1:
            raise RuntimeError('NumExpr rejected the configured thread count; restart the kernel.')
        os.environ['NUMEXPR_NUM_THREADS'] = str(active)
    return dict(detected_cpus=detected, maximum=maximum, active=active)

RECON_THREAD_CONFIG = configure_reconstruction_threads()

import yaml
import numpy as np
import h5py
import tifffile
import matplotlib.pyplot as plt
from scipy.ndimage import shift as image_shift
from skimage.registration import phase_cross_correlation
from IPython.display import display, HTML
try:
    import ipywidgets as widgets
except ImportError:
    widgets = None  # Every essential setting also has a plain Python value.


def require_compute_session(environ=None, hostname=None):
    """Reject recognizable NERSC login sessions before expensive operations.

    Slurm environment markers are a preliminary check, not proof of GPU access
    or the job's remaining time; verify those in the allocation terminal.
    """
    env = os.environ if environ is None else environ
    host = hostname if hostname is not None else (
        os.uname().nodename if hasattr(os, 'uname') else env.get('COMPUTERNAME', ''))
    host = host.lower().split('.')[0]
    on_nersc = bool(env.get('NERSC_HOST')) or host.startswith('perlmutter')
    if on_nersc:
        if 'login' in host or host.startswith('perlmutter'):
            raise RuntimeError('This appears to be a NERSC login node. Start a compute-node Jupyter server before reconstruction or denoising.')
        if not (env.get('SLURM_JOB_ID') or env.get('SLURM_JOBID')):
            raise RuntimeError('No Slurm job is visible in this NERSC kernel. Start compute-node Jupyter through JupyterHub and verify the allocation.')

def first_visible_gpu(environ=None):
    """Retain the first assigned CUDA identifier, including an explicit disabled mask."""
    env = os.environ if environ is None else environ
    visible = env.get('CUDA_VISIBLE_DEVICES')
    if visible is None:
        return '0'  # CUDA default; verify device access inside the allocation
    first = visible.split(',')[0].strip()
    return '' if first in ('', '-1', 'NoDevFiles') else first

def validate_gpu_ids(gpu_ids, environ=None):
    """Prevent the denoise CLI from widening an explicit scheduler GPU mask."""
    env = os.environ if environ is None else environ
    ids = [v.strip() for v in str(gpu_ids).split(',')]
    if not ids or any(v in ('', '-1', 'NoDevFiles') for v in ids) or len(set(ids)) != len(ids):
        raise ValueError('No valid GPU selection. Use an allocated GPU session and check CUDA_VISIBLE_DEVICES.')
    visible = env.get('CUDA_VISIBLE_DEVICES')
    if visible is not None:
        assigned = {v.strip() for v in visible.split(',') if v.strip()}
        if not set(ids).issubset(assigned) or first_visible_gpu(env) == '':
            raise ValueError('GPU_IDS must stay within the inherited CUDA_VISIBLE_DEVICES assignment.')
    return ','.join(ids)

def all_allocated_gpus(environ=None):
    """Select every inherited CUDA identifier without discovering unallocated devices."""
    env = os.environ if environ is None else environ
    visible = env.get('CUDA_VISIBLE_DEVICES')
    if visible is None:
        raise RuntimeError('CUDA_VISIBLE_DEVICES is unset. Start a GPU compute allocation with its scheduler GPU assignment visible to this kernel; do not guess device IDs from nvidia-smi.')
    return validate_gpu_ids(visible, env)

def select_output_root(override=None, environ=None):
    """Prefer NERSC scratch variables; never silently place large stacks in home."""
    if override is not None:
        root = Path(override).expanduser()
        if not root.is_absolute():
            raise ValueError('OUTPUT_ROOT_OVERRIDE must be an absolute scratch path.')
        return root.resolve()
    env = os.environ if environ is None else environ
    for name in ('PSCRATCH', 'SCRATCH'):
        if env.get(name):
            root = Path(env[name]).expanduser()
            if not root.is_absolute() or not root.is_dir():
                raise FileNotFoundError(f'{name} is not an accessible absolute scratch path: {root}')
            return root / 'noise2inverse360'
    als_scratch = Path('/alsuser/pscratch')
    if als_scratch.is_dir():
        return als_scratch / 'noise2inverse360'
    raise FileNotFoundError('No scratch location found. Set OUTPUT_ROOT_OVERRIDE to allocated scratch visible to this kernel.')

print('Kernel Python:', sys.executable)
print('Reconstruction NumExpr threads:', RECON_THREAD_CONFIG)
print('Host:', os.uname().nodename if hasattr(os, 'uname') else os.environ.get('COMPUTERNAME'))
print('NERSC_HOST:', os.environ.get('NERSC_HOST', '(unset)'),
      '| Slurm job:', os.environ.get('SLURM_JOB_ID') or os.environ.get('SLURM_JOBID') or '(unset)')
print('CUDA_VISIBLE_DEVICES:', os.environ.get('CUDA_VISIBLE_DEVICES', '(unset)'))

print('Python kernel ready. Widgets:', widgets is not None)

print("DONE: imports loaded; load helper cells next.")

In [ ]:
# 2. Define scan and run plan helpers

def digest(value):
    """Return a stable SHA-256 label for JSON-compatible settings or provenance."""
    return hashlib.sha256(json.dumps(value, sort_keys=True, allow_nan=False).encode()).hexdigest()

def fingerprint(path):
    """Identify an HDF5 source by path, file stat, data shape, type, and angles."""
    path = Path(path).resolve()
    stat = path.stat()
    with h5py.File(path, 'r') as f:
        data = f['/exchange/data']
        theta = np.asarray(f['/exchange/theta'][:])
        return dict(path=str(path), size=stat.st_size, mtime_ns=stat.st_mtime_ns,
                    shape=list(data.shape), dtype=str(data.dtype),
                    theta_sha256=hashlib.sha256(theta.tobytes()).hexdigest())

def validate_als_metadata(path):
    """Validate the metadata read by the modern ALS backend before expensive work."""
    with h5py.File(path, 'r') as f:
        nproj, nrows, ncols = f['/exchange/data'].shape
        expected = {
            '/measurement/instrument/detector/dimension_x': ncols,
            '/measurement/instrument/detector/dimension_y': nrows,
            '/process/acquisition/rotation/num_angles': nproj,
        }
        required = list(expected) + [
            '/process/acquisition/rotation/range',
            '/measurement/instrument/detector/pixel_size',
            '/measurement/instrument/monochromator/energy',
        ]
        for key in required:
            if key not in f:
                raise ValueError(f'ALS backend requires metadata: {key}')
            values = np.asarray(f[key][()])
            if values.ndim != 1 or not values.size or not np.isfinite(values).all():
                raise ValueError(f'ALS requires finite 1D metadata: {key}')
            if key in expected and values[0] != expected[key]:
                raise ValueError(f'ALS metadata disagrees with image dimensions: {key}')
        if f['/measurement/instrument/detector/pixel_size'][0] <= 0:
            raise ValueError('ALS attenuation scaling requires positive pixel size in mm.')
        theta = np.asarray(f['/exchange/theta'][:], dtype=np.float64)
        if theta.shape != (nproj,) or nproj < 2 or not np.isfinite(theta).all() or not np.all(np.diff(theta) > 0):
            raise ValueError('Expected one finite, increasing angle per projection.')
        span = float(theta[-1] - theta[0])
        stored = float(f['/process/acquisition/rotation/range'][0])
        if abs(stored - span) > max(1.0, 2 * float(np.median(np.diff(theta)))):
            raise ValueError('ALS angular range metadata disagrees with measured angles.')

def optional_camera_distance(path):
    """Report only the upstream second entry; never reinterpret a singleton distance."""
    key = '/measurement/instrument/camera_motor_stack/setup/camera_distance'
    with h5py.File(path, 'r') as f:
        if key not in f:
            return None
        values = np.asarray(f[key][()])
        if values.ndim != 1 or values.size < 2 or values.dtype.kind not in 'iuf':
            return None
        value = float(values[1])
        return value if np.isfinite(value) else None

def install_reconstruction_metadata_reader(functions):
    """Adapt unused distance metadata for this notebook's no-phase FBP/gridrec path.

    Changes only the privately imported module, never the checkout or raw HDF5.
    Missing distance remains None; no measurement or units are guessed.
    """
    original = functions.read_metadata
    def read_metadata(path, print_flag=True):
        if optional_camera_distance(path) is not None:
            return original(path, print_flag=print_flag)
        validate_als_metadata(path)
        with h5py.File(path, 'r') as f:
            def first(key):
                return f[key][0]
            metadata = dict(
                numslices=int(first('/measurement/instrument/detector/dimension_y')),
                numrays=int(first('/measurement/instrument/detector/dimension_x')),
                pxsize=float(first('/measurement/instrument/detector/pixel_size')) / 10.0,
                numangles=int(first('/process/acquisition/rotation/num_angles')),
                propagation_dist=None,
                kev=float(first('/measurement/instrument/monochromator/energy')) / 1000.0,
                angularrange=float(first('/process/acquisition/rotation/range')))
        if print_flag:
            print(Path(path).name, metadata)
            print('Camera distance unavailable; unused in this no-phase FBP/gridrec workflow.')
        return metadata
    functions.read_metadata = read_metadata

def inspect_scan(path):
    """Check the ALS-style 180-degree HDF5 layout and return readable metadata."""
    with h5py.File(path, 'r') as f:
        required = ['data', 'data_white', 'data_dark', 'theta']
        for name in required:
            assert '/exchange/' + name in f, f'Missing /exchange/{name}'
        shape = f['/exchange/data'].shape
        if len(shape) != 3 or not all(n > 0 for n in shape) or shape[0] < 4:
            raise ValueError('Expected [projection, row, column] with at least four projections for even/odd views.')
        for name in ('data_white', 'data_dark'):
            calibration = f['/exchange/' + name]
            if calibration.ndim != 3 or calibration.shape[0] == 0 or calibration.shape[1:] != shape[1:]:
                raise ValueError(f'{name} must contain nonempty [frame, row, column] images matching the scan.')
        validate_als_metadata(path)
        theta = np.asarray(f['/exchange/theta'][:], dtype=np.float64)
        assert theta.shape == (shape[0],) and np.isfinite(theta).all()
        assert np.all(np.diff(theta) > 0), 'This workflow requires increasing projection angles.'
        # ALS dxfile theta is in degrees. This workflow handles standard 180° scans.
        step = float(np.median(np.diff(theta)))
        assert abs(float(theta[-1] - theta[0]) - 180) <= max(1, 2 * step), \
            'Expected an ALS 180-degree scan. Check angle units/geometry before proceeding.'
        def scalar(key):
            if key not in f:
                return None
            values = np.asarray(f[key][()])
            if values.ndim != 1 or not values.size:
                raise ValueError(f'Expected nonempty 1D metadata: {key}')
            val = float(values[0])
            return val if np.isfinite(val) else None
        pixel_mm = scalar('/measurement/instrument/detector/pixel_size')
        energy_ev = scalar('/measurement/instrument/monochromator/energy')
        info = dict(shape=list(shape), dtype=str(f['/exchange/data'].dtype),
                    flats=int(f['/exchange/data_white'].shape[0]),
                    darks=int(f['/exchange/data_dark'].shape[0]),
                    angle_start_deg=float(theta[0]), angle_end_deg=float(theta[-1]),
                    angle_step_deg=step,
                    pixel_um=pixel_mm * 1000 if pixel_mm and pixel_mm > 0 else None,
                    distance_mm=optional_camera_distance(path),
                    stored_energy_ev=energy_ev,
                    energy_kev=energy_ev / 1000 if energy_ev and energy_ev > 0 else None)
        if info['distance_mm'] is None:
            print('Camera distance has no usable second entry; recorded as unknown. This workflow does not use phase retrieval or camera distance.')
        return info

def run_logged(parts, log_path, cwd):
    """Run an external command, stream its output, and save a complete log.

    The command is an argument list, not a shell string. An interrupt stops
    the subprocess group so a training job does not keep running unseen.
    """
    cmd = [str(v) for v in parts]
    if any(v in ('train', 'slice', 'volume') for v in cmd) and not any(v in ('-h', '--help') for v in cmd):
        require_compute_session()
    env = os.environ.copy()
    env['MPLBACKEND'] = 'Agg'
    resolved = shutil.which(cmd[0]) or cmd[0]
    if Path(resolved).is_file():
        env['PATH'] = str(Path(resolved).resolve().parent) + os.pathsep + env.get('PATH', '')
    env['PYTHONUNBUFFERED'] = '1'
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print(shlex.join(cmd))
    print('Complete log:', log_path)
    tail = deque(maxlen=14)
    screen = display({'text/plain': 'Starting…'}, raw=True, display_id=True)
    with log_path.open('a', encoding='utf-8') as log:
        log.write('\n' + datetime.now(timezone.utc).isoformat() + '\n' + shlex.join(cmd) + '\n')
        proc = subprocess.Popen(cmd, cwd=str(cwd), env=env, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, errors='replace',
                                bufsize=1, start_new_session=(os.name == 'posix'))
        last_update = 0
        try:
            for line in proc.stdout:
                log.write(line)
                log.flush()
                tail.append(line.rstrip())
                if time.monotonic() - last_update > 1:
                    screen.update({'text/plain': '\n'.join(tail)}, raw=True)
                    last_update = time.monotonic()
            result = proc.wait()
        except BaseException:
            if proc.poll() is None:
                if os.name == 'posix':
                    os.killpg(proc.pid, signal.SIGTERM)
                else:
                    proc.terminate()
                try:
                    proc.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    if os.name == 'posix':
                        os.killpg(proc.pid, signal.SIGKILL)
                    else:
                        proc.kill()
                    proc.wait()
            raise
        finally:
            proc.stdout.close()
            screen.update({'text/plain': '\n'.join(tail)}, raw=True)
    if result:
        raise subprocess.CalledProcessError(result, cmd)

def tiff_paths(folder):
    """List TIFF images directly in one reconstruction or inference folder."""
    return sorted(p for p in Path(folder).iterdir()
                  if p.is_file() and p.suffix.lower() in ('.tif', '.tiff'))

def utc_now():
    return datetime.now(timezone.utc).isoformat()


def json_write(path, value):
    """Write JSON atomically through a unique temporary file."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name('.' + path.name + '.' + uuid.uuid4().hex + '.tmp')
    try:
        with temp.open('x', encoding='utf-8') as stream:
            json.dump(value, stream, indent=2, allow_nan=False)
            stream.flush()
            os.fsync(stream.fileno())
        temp.replace(path)
    finally:
        temp.unlink(missing_ok=True)


@contextlib.contextmanager
def exclusive_lock(path):
    """Acquire an operation lock and release only the token owned by this context."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    token = dict(token=uuid.uuid4().hex, pid=os.getpid(), host=socket.gethostname(),
                 job=os.environ.get('SLURM_JOB_ID'), created_utc=utc_now())
    try:
        stream = path.open('x', encoding='utf-8')
    except FileExistsError as error:
        raise RuntimeError(f'Active or stale lock: {path}. Confirm the original job/kernel stopped before explicit repair.') from error
    try:
        with stream:
            json.dump(token, stream)
        yield token
    finally:
        if path.exists() and json.loads(path.read_text(encoding='utf-8')) == token:
            path.unlink()


def invalidate_downstream():
    for name in ('AUTO_COR_RESULT', 'MANUAL_COR_SELECTION', 'plan', 'active_plan',
                 'FULL_DIR', 'EVEN_DIR', 'ODD_DIR', 'stack_lists', 'CONFIG_YAML',
                 'TRAIN_DIR', 'SELECTED_MODE', 'VOLUME_DIR', 'CHECKPOINT',
                 'TRAINING_CONTEXT', 'PAIR_CONTEXT', 'CENTER_ESTIMATE', 'center_control',
                 'CLEANUP_PREVIEW', 'TRAINING_HISTORY'):
        globals().pop(name, None)


def select_context(source, run_dir, info):
    invalidate_downstream()
    globals()['WORKFLOW_CONTEXT'] = dict(source=copy.deepcopy(source),
        run_dir=str(Path(run_dir).resolve()), dimensions=list(info['shape']),
        backend=None, settings=None, generation=uuid.uuid4().hex)
    return copy.deepcopy(WORKFLOW_CONTEXT)


def configure_context(backend, settings):
    context = globals().get('WORKFLOW_CONTEXT')
    if context is None:
        raise RuntimeError('Inspect the selected scan first.')
    if context['backend'] != backend or context['settings'] != settings:
        invalidate_downstream()
        context.update(backend=copy.deepcopy(backend), settings=copy.deepcopy(settings),
                       generation=uuid.uuid4().hex)
    return context_token()


def context_token():
    context = globals().get('WORKFLOW_CONTEXT')
    if not context or context['backend'] is None:
        raise RuntimeError('Inspect the scan and configure the backend first.')
    return copy.deepcopy(context)


def assert_context(captured, check_source=False):
    if captured != context_token():
        raise ValueError('Stale controls/results: rerun this action for the selected scan.')
    picker = globals().get('file_picker')
    if picker is not None and str(Path(picker.value).resolve()) != captured['source']['path']:
        raise ValueError('Scan selection changed; inspect the selected scan first.')
    if picker is None and 'default_scan' in globals() and str(Path(default_scan).resolve()) != captured['source']['path']:
        raise ValueError('Numeric/path scan selection changed; inspect it first.')
    if 'RAW_H5' in globals() and str(Path(RAW_H5).resolve()) != captured['source']['path']:
        raise ValueError('Selected raw source changed; inspect it first.')
    if 'RUN_LABEL' in globals() and Path(captured['run_dir']).name != RUN_LABEL:
        raise ValueError('Run label changed; inspect the selected run first.')
    if globals().get('RECON_SETTINGS', captured['settings']) != captured['settings']:
        raise ValueError('Reconstruction settings changed; configure the backend again.')
    if globals().get('BACKEND_RECORD', captured['backend']) != captured['backend']:
        raise ValueError('Backend changed; configure it again.')
    if str(Path(globals().get('RUN_DIR', captured['run_dir'])).resolve()) != captured['run_dir']:
        raise ValueError('Run selection changed; inspect it again.')
    if check_source and fingerprint(captured['source']['path']) != captured['source']:
        raise ValueError('Raw source changed; inspect the scan again.')


def path_is_within(path, root):
    """Containment check compatible with Python versions before 3.9."""
    path, root = Path(path), Path(root)
    return path == root or root in path.parents


def safe_child(root, name):
    root = Path(root).resolve()
    if not isinstance(name, str) or not name or Path(name).name != name or name in ('.', '..'):
        raise ValueError('Output names must be simple directory names.')
    path = root / name
    if path.resolve() == root or not path_is_within(path.resolve(), root):
        raise ValueError('Output escapes its run directory.')
    return path


def is_link_like(path):
    """Detect junctions on Python versions that lack Path.is_junction()."""
    path = Path(path)
    if path.is_symlink() or (hasattr(path, 'is_junction') and path.is_junction()):
        return True
    try:
        tag = getattr(path.lstat(), 'st_reparse_tag', 0)
        return tag in (getattr(stat, 'IO_REPARSE_TAG_SYMLINK', -1),
                       getattr(stat, 'IO_REPARSE_TAG_MOUNT_POINT', -2))
    except FileNotFoundError:
        return False


def validate_plan(saved, plan_path=None):
    if not isinstance(saved, dict) or saved.get('schema') != 4:
        raise ValueError('Unsupported run-plan format. Create a new v3 run with a new RUN_LABEL.')
    rec = saved['reconstruction']
    if rec['geometry'] != dict(center_convention='midpoint_offset', scan_degrees=180,
                               angle_order='increasing', subsets='disjoint_even_odd'):
        raise ValueError('Unsupported reconstruction geometry.')
    shape = rec['dimensions']
    if len(shape) != 3 or any(type(n) is not int or n <= 0 for n in shape) or shape[0] < 4 or shape[2] % 2:
        raise ValueError('Invalid frozen dimensions.')
    if rec['source']['shape'] != shape or not np.isfinite(rec['center']) or abs(rec['center']) >= shape[2]/2:
        raise ValueError('Invalid frozen source or midpoint COR.')
    if rec['backend']['name'] != 'als_modern' or rec['settings']['method'] not in ('fbp', 'gridrec'):
        raise ValueError('Unsupported backend/settings.')
    root = Path(saved['run_dir'])
    if not root.is_absolute() or (plan_path and Path(plan_path).resolve().parent != root.resolve()):
        raise ValueError('Plan location disagrees with its run directory.')
    if set(rec['outputs']) != {'full', 'even', 'odd'} or len(set(rec['outputs'].values())) != 3:
        raise ValueError('Expected three distinct reconstruction output directories.')
    for name in rec['outputs'].values():
        safe_child(root, name)
    if saved['provenance'].get('notebook_version') not in ('3.0', '3.1', '3.2', '3.3'):
        raise ValueError('Unsupported release provenance. Create a new v3 run.')
    return saved


def read_plan(path):
    path = Path(path)
    return validate_plan(json.loads(path.read_text(encoding='utf-8')), path)


def reconstruction_identity(saved):
    # Selection notes, timestamps, metadata and run paths do not enter stage identity.
    return copy.deepcopy(validate_plan(saved)['reconstruction'])


def stage_signature(saved, stage):
    if stage not in ('full', 'even', 'odd'):
        raise ValueError('Unknown reconstruction stage.')
    nproj = saved['reconstruction']['dimensions'][0]
    subset = dict(kind=stage, start={'full': 0, 'even': 0, 'odd': 1}[stage],
                  step=1 if stage == 'full' else 2, stop=nproj)
    return digest(dict(reconstruction=reconstruction_identity(saved), subset=subset,
                       output_format='float32_row_tiff_v1', completion_schema=1))


def resolve_center(captured, manual=None, numeric=None, existing=None, automatic=None,
                   policy='auto_if_eligible'):
    assert_context(captured, check_source=True)
    if policy not in ('auto_if_eligible', 'manual_only'):
        raise ValueError('COR_POLICY must be auto_if_eligible or manual_only.')
    if numeric is not None:
        decision = dict(center=float(numeric), method='numeric_override', context=captured)
    elif manual is not None:
        assert_context(manual['context'])
        decision = copy.deepcopy(manual)
    elif existing is not None:
        rec = validate_plan(existing)['reconstruction']
        if (rec['source'] != captured['source'] or rec['backend'] != captured['backend'] or
                rec['settings'] != captured['settings'] or existing['run_dir'] != captured['run_dir']):
            raise ValueError('Frozen run identity changed. Use a new RUN_LABEL.')
        return float(rec['center']), copy.deepcopy(existing['selection'])
    elif policy == 'auto_if_eligible':
        center = checked_auto_center(automatic, captured)
        decision = dict(center=center, method='tomopy_find_center_vo', result=copy.deepcopy(automatic))
    else:
        raise ValueError('Manual-only policy: apply a reviewed COR or enter COR_OVERRIDE in pixels.')
    center = decision['center']
    if not np.isfinite(center) or abs(center) >= captured['dimensions'][2]/2:
        raise ValueError('COR must be finite and strictly within detector midpoint bounds.')
    return float(center), decision


def freeze_run(captured, info, center, selection):
    assert_context(captured, check_source=True)
    root = Path(captured['run_dir'])
    stem = Path(captured['source']['path']).stem
    saved = dict(schema=4, run_dir=str(root), reconstruction=dict(
        source=copy.deepcopy(captured['source']), dimensions=captured['dimensions'],
        geometry=dict(center_convention='midpoint_offset', scan_degrees=180,
                      angle_order='increasing', subsets='disjoint_even_odd'),
        center=float(center), backend=copy.deepcopy(captured['backend']),
        settings=copy.deepcopy(captured['settings']),
        outputs=dict(full=stem+'_rec', even=stem+'_rec_0', odd=stem+'_rec_1')),
        selection=copy.deepcopy(selection), provenance=dict(notebook_version='3.3',
        policy_version='vo_rows_v1', frozen_utc=utc_now(), scan_metadata=info))
    validate_plan(saved)
    with exclusive_lock(root / '.run.lock'):
        path = root / 'run_plan.json'
        if path.exists():
            old = read_plan(path)
            if reconstruction_identity(old) != reconstruction_identity(saved):
                raise ValueError('Frozen reconstruction identity changed. Use a new RUN_LABEL.')
            return old
        forbidden = list(root.glob('*_complete.json')) + list(root.glob('*_command.json'))
        forbidden += [root / name for name in saved['reconstruction']['outputs'].values() if (root / name).exists()]
        if forbidden or (root / 'TrainOutput').exists():
            raise ValueError('Prior outputs exist without a v3 plan. Create a new v3 run directory.')
        json_write(path, saved)
    return saved

print("DONE: scan, state and plan helpers loaded.")

In [ ]:
# 3. Define reconstruction and zoom display helpers

_als_cache = {}
def load_als(backend):
    """Import the exact modern helper and its sibling functions from the saved checkout."""
    assert backend['name'] == 'als_modern', 'Legacy backend plan: select a new run label.'
    directory = Path(backend['directory'])
    for name, expected in backend['source_sha256'].items():
        assert hashlib.sha256((directory / name).read_bytes()).hexdigest() == expected, \
            'ALS source changed since this plan was frozen.'
    key = digest(backend)
    if key not in _als_cache:
        def import_file(name):
            spec = importlib.util.spec_from_file_location(name, directory / (name + '.py'))
            module = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(module)
            return module
        functions = import_file('ALS_recon_functions')
        install_reconstruction_metadata_reader(functions)
        previous = sys.modules.get('ALS_recon_functions')
        sys.modules['ALS_recon_functions'] = functions
        try:
            helper = import_file('ALS_recon_helper')
        finally:
            if previous is None:
                sys.modules.pop('ALS_recon_functions', None)
            else:
                sys.modules['ALS_recon_functions'] = previous
        assert helper.als is functions
        _als_cache[key] = helper
    return _als_cache[key]

def als_chunk(source, settings, center, start, stop, backend):
    """Call the ALS_recon notebook's modern reconstruction API for rows [start, stop)."""
    if settings['method'] not in ('fbp', 'gridrec'):
        raise ValueError('The optional-distance adapter is restricted to FBP/gridrec without phase retrieval.')
    require_compute_session()
    validate_als_metadata(source)
    helper = load_als(backend)
    with h5py.File(source, 'r') as f:
        nproj, rows, columns = f['/exchange/data'].shape
        expected_angles = -np.deg2rad(f['/exchange/theta'][:])
    assert 0 <= start < stop <= rows
    # Check the installed DXchange angle convention before reconstructing.
    # proj=None intentionally reads every angle in this original/virtual input.
    _, angles = helper.als.read_data(str(source), proj=None, sino=slice(start, start+1, 1),
        preprocess_settings=settings['preprocessing'], postprocess_settings=settings['postprocessing'])
    np.testing.assert_allclose(angles, expected_angles, atol=1e-6)
    rec, _ = helper.reconstruct(
        path=str(source), angles_ind=None, slices_ind=slice(start, stop, 1),
        COR=float(center), method=settings['method'], proj_downsample=1, fc=settings['fc'],
        preprocessing_settings=copy.deepcopy(settings['preprocessing']),
        postprocessing_settings=copy.deepcopy(settings['postprocessing']),
        mask=True, convert360to180=False, use_gpu=settings['use_gpu'])
    rec = np.asarray(rec, dtype=np.float32)
    assert rec.shape == (stop-start, columns, columns), 'Unexpected ALS row count or image shape.'
    assert np.isfinite(rec).all(), 'Nonfinite ALS reconstruction; inspect calibration and processing.'
    return rec

def run_als_stage(source, output, settings, center, nrows, backend, log_path):
    """Write an ALS TIFF stack in row chunks with per-image atomic replacement."""
    output, log_path = Path(output), Path(log_path)
    output.mkdir(parents=True, exist_ok=True)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open('a', encoding='utf-8') as log:
        log.write('\nALS stage ' + datetime.now(timezone.utc).isoformat() + '\n')
        for start in range(0, nrows, settings['als_chunk_sino']):
            stop = min(nrows, start + settings['als_chunk_sino'])
            # Recompute this chunk on a stage retry; writes replace only this run's tracked TIFFs.
            with contextlib.redirect_stdout(log), contextlib.redirect_stderr(log):
                rec = als_chunk(source, settings, center, start, stop, backend)
            for offset, image in enumerate(rec):
                target = output / f'recon_{start+offset:05d}.tiff'
                temp = target.with_suffix('.part')  # Must not match Noise2Inverse's *.tif* glob
                tifffile.imwrite(temp, image)
                temp.replace(target)
            del rec
            log.write(f'Wrote rows {start}:{stop}\n')
            log.flush()
            print(f'{output.name}: wrote rows {start}:{stop} / {nrows}', flush=True)

# Optional starting zoom/pan for image displays; coordinates are (x, y) pixels.
IMAGE_VIEW_OPTIONS = {'zoom': 1., 'center': None}

def image_zoom_bounds(shape, zoom=1., center=None):
    """Return exclusive pixel bounds for a zoomed viewport, without resampling."""
    if len(shape) != 2 or min(shape) < 1:
        raise ValueError('The viewer needs a nonempty two-dimensional image.')
    height, width = map(int, shape)
    zoom = float(zoom)
    if not np.isfinite(zoom) or zoom < 1:
        raise ValueError('Zoom must be finite and at least 1.')
    center = ((width-1)/2, (height-1)/2) if center is None else tuple(center)
    if len(center) != 2 or not np.isfinite(center).all():
        raise ValueError('Center must be finite (x, y) pixel coordinates.')
    crop_width = max(1, int(round(width/zoom)))
    crop_height = max(1, int(round(height/zoom)))
    x0 = min(width-crop_width, max(0, int(round(center[0]-(crop_width-1)/2))))
    y0 = min(height-crop_height, max(0, int(round(center[1]-(crop_height-1)/2))))
    return x0, x0+crop_width, y0, y0+crop_height


def image_display_limits(image):
    """Set a full-image contrast window that remains fixed across crops."""
    finite = np.asarray(image)[np.isfinite(image)]
    if not finite.size:
        raise ValueError('The image contains no finite pixels.')
    low, high = map(float, np.percentile(finite, [1,99]))
    if high <= low:
        high = low + max(1., abs(low)*1e-6)
    return low, high


def image_view_png(image, bounds, limits, cmap='gray', overlay=None):
    """Encode only the selected native pixels; keep NaN borders transparent."""
    x0, x1, y0, y1 = bounds
    low, high = map(float, limits)
    if not np.isfinite([low,high]).all() or high <= low:
        raise ValueError('White level must be greater than black level; both must be finite.')
    crop = np.asarray(image)[y0:y1,x0:x1]
    rgba = plt.get_cmap(cmap)(np.ma.masked_invalid((crop-low)/(high-low)), bytes=True)
    if overlay is not None:
        over_image, over_cmap, opacity = overlay
        over_crop = np.asarray(over_image)[y0:y1,x0:x1]
        over_rgba = plt.get_cmap(over_cmap)(np.ma.masked_invalid((over_crop-low)/(high-low)),bytes=True)
        alpha = np.where(np.isfinite(over_crop), float(opacity), 0.)[...,None]
        rgba[...,:3] = np.round(rgba[...,:3]*(1-alpha)+over_rgba[...,:3]*alpha).astype(np.uint8)
    stream = io.BytesIO()
    Image.fromarray(rgba).save(stream, format='PNG')
    return stream.getvalue()


def zoom_image_viewer(images, titles=None, limits=None, cmaps=None, overlays=None,
                      captured=None, title='Image inspection', zoom=None, center=None,
                      load_images=None, count=None, index=0, index_label='Slice'):
    """Linked crop/contrast controls for arrays or a lazily loaded TIFF slice.

    Zoom and center coordinates affect display only. No image is binned or modified.
    The viewer retains only the current slice, and callbacks verify their run context.
    """
    defaults = globals().get('IMAGE_VIEW_OPTIONS', {})
    zoom = float(defaults.get('zoom',1.) if zoom is None else zoom)
    center = defaults.get('center') if center is None else center
    arrays = [np.asarray(image) for image in images]
    if not arrays or any(a.ndim != 2 or a.shape != arrays[0].shape or not np.isrealobj(a) for a in arrays):
        raise ValueError('Viewer images must be real, matching two-dimensional arrays.')
    for array in arrays:
        if not np.isfinite(array).any():
            raise ValueError('Viewer images must contain finite pixels.')
    shape = arrays[0].shape
    bounds = image_zoom_bounds(shape,zoom,center)
    number = len(arrays)
    titles = list(titles) if titles is not None else [f'Image {i+1}' for i in range(number)]
    cmaps = list(cmaps) if cmaps is not None else ['gray']*number
    limits = [tuple(map(float,pair)) for pair in limits] if limits is not None else [image_display_limits(arrays[0])]*number
    overlays = list(overlays) if overlays is not None else [None]*number
    if any(len(items) != number for items in (titles,cmaps,limits,overlays)):
        raise ValueError('Titles, windows, color maps, and overlays must match the image count.')
    for pair in limits:
        if len(pair) != 2 or not np.isfinite(pair).all() or pair[1] <= pair[0]:
            raise ValueError('Each contrast window needs finite black < white levels.')
    for overlay in overlays:
        if overlay is not None and (len(overlay) != 3 or np.shape(overlay[0]) != shape or
                                   not np.isfinite(overlay[2]) or not 0 <= overlay[2] <= 1):
            raise ValueError('Each overlay needs a matching image, color map and opacity in [0,1].')
    if count is not None and (type(count) is not int or count < 1 or not callable(load_images) or not 0 <= index < count):
        raise ValueError('A slice browser needs a loader, a positive count and a valid initial index.')
    if captured is not None:
        assert_context(captured)

    # Full-image windows stay constant through zoom, pan and slice changes.
    groups, group_keys, panel_groups = [], [], []
    for i, (pair,cmap) in enumerate(zip(limits,cmaps)):
        key = (pair,cmap)
        if key not in group_keys:
            group_keys.append(key)
            groups.append(dict(window=pair, initial=pair, titles=[]))
        group = group_keys.index(key)
        groups[group]['titles'].append(titles[i])
        panel_groups.append(group)
    state = dict(images=arrays,index=index,bounds=bounds,groups=groups,error=None)

    if not widgets:
        fig, axes = plt.subplots(1,number,figsize=(7*number,7),squeeze=False)
        x0, x1, y0, y1 = bounds
        extent = (x0-.5,x1-.5,y1-.5,y0-.5)
        try:
            for i,ax in enumerate(axes[0]):
                ax.imshow(arrays[i][y0:y1,x0:x1],cmap=cmaps[i],vmin=limits[i][0],vmax=limits[i][1],
                          interpolation='nearest',origin='upper',extent=extent)
                if overlays[i] is not None:
                    over, over_cmap, opacity = overlays[i]
                    ax.imshow(np.asarray(over)[y0:y1,x0:x1],cmap=over_cmap,alpha=opacity,
                              vmin=limits[i][0],vmax=limits[i][1],interpolation='nearest',origin='upper',extent=extent)
                ax.set_title(titles[i]); ax.set_xlabel('x (pixel)'); ax.set_ylabel('y (pixel)')
            fig.suptitle(title)
            fig.subplots_adjust(left=.07,right=.98,bottom=.12,top=.85,wspace=.25)
            plt.show()
        finally:
            plt.close(fig)
        print('Static pixel view. Enable ipywidgets for zoom controls, or set IMAGE_VIEW_OPTIONS =',
              "{'zoom': 8, 'center': (x, y)} and rerun the display cell.")
        return dict(state=state,widget=None,controls={})

    height, width = shape
    start_center = ((width-1)//2,(height-1)//2) if center is None else center
    layout = widgets.Layout
    zoom_control = widgets.FloatSlider(value=min(zoom,64.),min=1.,max=64.,step=1.,
        description='Zoom:',readout_format='.0f',continuous_update=False,layout=layout(width='330px'))
    x_control = widgets.IntSlider(value=int(np.clip(round(start_center[0]),0,width-1)),min=0,max=width-1,
        description='Center x:',continuous_update=False,layout=layout(width='330px'))
    y_control = widgets.IntSlider(value=int(np.clip(round(start_center[1]),0,height-1)),min=0,max=height-1,
        description='Center y:',continuous_update=False,layout=layout(width='330px'))
    reset = widgets.Button(description='Reset view',icon='expand')
    directions = [widgets.Button(description=label,layout=layout(width='65px')) for label in ('←','→','↑','↓')]
    group_control = widgets.Dropdown(options=[(' / '.join(g['titles']),i) for i,g in enumerate(groups)],
        value=0,description='Contrast:',layout=layout(width='350px'))
    low_control = widgets.FloatText(value=groups[0]['window'][0],description='Black:',layout=layout(width='235px'))
    high_control = widgets.FloatText(value=groups[0]['window'][1],description='White:',layout=layout(width='235px'))
    reset_contrast = widgets.Button(description='Reset contrast')
    panels = [widgets.HTML(layout=layout(flex='1 1 300px',min_width='280px')) for _ in arrays]
    status = widgets.HTML()
    index_control = None if count is None else widgets.IntSlider(value=index,min=0,max=count-1,
        description=index_label+':',continuous_update=False,layout=layout(width='550px'))
    controls = dict(zoom=zoom_control,x=x_control,y=y_control,reset=reset,
                    contrast_group=group_control,black=low_control,white=high_control,
                    reset_contrast=reset_contrast,index=index_control,pan=directions)
    muted = [False]

    def render(change=None):
        if muted[0]:
            return
        try:
            if captured is not None:
                assert_context(captured)
            selected = index if index_control is None else index_control.value
            new_arrays = state['images']
            if selected != state['index']:
                new_arrays = [np.asarray(a) for a in load_images(selected)]
                if (len(new_arrays) != number or any(a.shape != shape or not np.isrealobj(a) or
                    not np.isfinite(a).any() for a in new_arrays)):
                    raise ValueError('Selected slice has invalid or mismatched image pixels.')
            window = (float(low_control.value),float(high_control.value))
            if not np.isfinite(window).all() or window[1] <= window[0]:
                raise ValueError('Set finite contrast levels with White greater than Black.')
            new_bounds = image_zoom_bounds(shape,zoom_control.value,(x_control.value,y_control.value))
            group = group_control.value
            windows = [window if i == group else g['window'] for i,g in enumerate(groups)]
            # Finish encoding every panel before replacing any of the comparison views.
            payloads = []
            for i,array in enumerate(new_arrays):
                png = image_view_png(array,new_bounds,windows[panel_groups[i]],cmaps[i],overlays[i])
                encoded = base64.b64encode(png).decode('ascii')
                label = html.escape(str(titles[i]))
                low,high = windows[panel_groups[i]]
                payloads.append(f'<div style="border:1px solid #dbe2ea;border-radius:8px;padding:8px;background:#f8fafc">'
                    f'<b>{label}</b><div style="font-size:11px;color:#58677a;margin:4px 0">'
                    f'Black {low:.5g} · White {high:.5g}</div>'
                    f'<img alt="{label}" src="data:image/png;base64,{encoded}" '
                    'style="display:block;width:100%;height:auto;max-height:620px;object-fit:contain;'
                    'image-rendering:crisp-edges;image-rendering:pixelated;background:#17212f"></div>')
            state.update(images=new_arrays,index=selected,bounds=new_bounds,error=None)
            groups[group]['window'] = window
            for panel,payload in zip(panels,payloads):
                panel.value = payload
            x0,x1,y0,y1 = new_bounds
            prefix = '' if index_control is None else f'{html.escape(index_label)} {selected} · '
            status.value = (f'<div style="color:#42536a;padding:6px 0">{prefix}'
                f'{zoom_control.value:g}× zoom · x {x0}–{x1-1}, y {y0}–{y1-1} · '
                f'{x1-x0} × {y1-y0} source pixels. Zoom and center controls are linked across panels.</div>')
        except Exception as error:
            state['error'] = str(error)
            status.value = '<b style="color:#b42318">Viewer: '+html.escape(str(error))+'</b>'

    def choose_group(change):
        muted[0] = True
        try:
            low_control.value,high_control.value = groups[group_control.value]['window']
        finally:
            muted[0] = False
        render()

    def reset_view(button):
        muted[0] = True
        try:
            zoom_control.value = 1.
            x_control.value,y_control.value = (width-1)//2,(height-1)//2
        finally:
            muted[0] = False
        render()

    def reset_windows(button):
        for group in groups:
            group['window'] = group['initial']
        choose_group(None)

    def pan(dx,dy):
        x0,x1,y0,y1 = state['bounds']
        muted[0] = True
        try:
            x_control.value = int(np.clip(x_control.value+dx*max(1,(x1-x0)//4),0,width-1))
            y_control.value = int(np.clip(y_control.value+dy*max(1,(y1-y0)//4),0,height-1))
        finally:
            muted[0] = False
        render()

    for control in (zoom_control,x_control,y_control,low_control,high_control,index_control):
        if control is not None:
            control.observe(render,names='value')
    group_control.observe(choose_group,names='value')
    reset.on_click(reset_view); reset_contrast.on_click(reset_windows)
    for button,(dx,dy) in zip(directions,((-1,0),(1,0),(0,-1),(0,1))):
        button.on_click(lambda button,dx=dx,dy=dy:pan(dx,dy))
    wrap = layout(display='flex',flex_flow='row wrap',align_items='center',width='100%')
    rows = [widgets.HTML(value='<b>'+html.escape(title)+'</b>')]
    if index_control is not None:
        rows.append(index_control)
    rows.extend([widgets.Box([zoom_control,x_control,y_control],layout=wrap),
        widgets.Box([reset]+directions,layout=wrap),
        widgets.Box([group_control,low_control,high_control,reset_contrast],layout=wrap),
        status,widgets.Box(panels,layout=layout(display='flex',flex_flow='row wrap',width='100%'))])
    viewer = widgets.VBox(rows,layout=layout(width='100%'))
    render()
    display(viewer)
    return dict(state=state,widget=viewer,controls=controls,panels=panels,status=status)


def browse_tiffs(paths, title='Slice browser'):
    """Read one TIFF at a time and retain the same zoom/window as slices change."""
    captured = context_token()
    paths = tuple(Path(p) for p in paths)
    if not paths:
        raise ValueError('No TIFF files found.')
    index = len(paths)//2
    sample = tifffile.imread(paths[index])
    def load(index, captured=captured, paths=paths):
        assert_context(captured)
        return [tifffile.imread(paths[index])]
    return zoom_image_viewer([sample],titles=[title],limits=[image_display_limits(sample)],
        captured=captured,title=title,load_images=load,count=len(paths),index=index)


print("DONE: reconstruction and display helpers loaded.")

In [ ]:
# 4. Define projection alignment helpers

def mean_frames(dset):
    """Average calibration frames one at a time to limit peak memory."""
    total = np.zeros(dset.shape[1:], dtype=np.float64)
    for i in range(dset.shape[0]):
        total += dset[i]
    return (total / dset.shape[0]).astype(np.float32)

def alignment_projections(path, bright_ratio=1.0):
    """Flat/dark-correct the first and opposite raw projections for alignment."""
    with h5py.File(path, 'r') as f:
        angles = f['/exchange/theta'][:]
        opposite = int(np.argmin(np.abs(angles - angles[0] - 180)))
        assert abs(float(angles[opposite] - angles[0]) - 180) < 1
        flat = mean_frames(f['/exchange/data_white'])
        dark = mean_frames(f['/exchange/data_dark'])
        denominator = flat - dark
        good = denominator > 0
        images = []
        for index in (0, opposite):
            raw = f['/exchange/data'][index].astype(np.float32)
            transmission = np.ones_like(raw)
            np.divide((raw - dark) * bright_ratio, denominator, out=transmission, where=good)
            transmission = np.clip(transmission, 1e-6, None)
            images.append(-np.log(transmission))
    return images[0], images[1], opposite

def center_estimate(first, opposite):
    """Estimate the midpoint-offset rotation center from reflected projection shift."""
    mirrored = opposite[:, ::-1]
    displacement, _, _ = phase_cross_correlation(first, mirrored, upsample_factor=10)
    # Same pixel-index convention as TomoPy find_center_pc.
    return (float(displacement[1]) - 1) / 2, displacement

def alignment_figure(first, opposite, center):
    """Inspect the first/opposite overlay and residual with linked zoom controls."""
    dx = 2 * center + 1
    moved = image_shift(opposite[:, ::-1], (0, dx), order=1,
                        mode='constant', cval=np.nan, prefilter=False)
    difference = first - moved
    lo, hi = image_display_limits(first)
    span = max(float(hi-lo)*.15, 1e-6)
    return zoom_image_viewer([first,difference],
        titles=[f'Overlay — COR offset {center:.3f}', 'First − shifted reflected projection'],
        limits=[(lo,hi),(-span,span)],cmaps=['gray','coolwarm'],
        overlays=[(moved,'magma',.4),None],captured=context_token(),title='Projection alignment')

print("DONE: projection alignment helpers loaded.")

In [ ]:
# 5. Define Vo COR and review helpers

def vo_options(options=None):
    """Native Vo parameters and provisional multi-row agreement checks."""
    opts = dict(smin=-50, smax=50, srad=6., step=.1, ratio=.5, drop=20,
                ncore=1, disagreement_pixels=1.)
    supplied = {} if options is None else dict(options)
    unknown = set(supplied) - set(opts)
    if unknown:
        raise ValueError('Unknown Vo options: ' + ', '.join(sorted(unknown)))
    opts.update(supplied)
    for name in ('smin', 'smax', 'drop', 'ncore'):
        if type(opts[name]) is not int:
            raise ValueError(f'{name} must be an integer.')
    if opts['smin'] >= opts['smax'] or opts['drop'] < 0 or opts['ncore'] < 1:
        raise ValueError('Use smin < smax, drop >= 0 and ncore >= 1.')
    for name in ('srad', 'step', 'ratio', 'disagreement_pixels'):
        if not np.isfinite(opts[name]) or opts[name] <= 0:
            raise ValueError(f'{name} must be finite and positive.')
    if opts['srad'] < 1 or not .1 <= opts['step'] <= opts['srad']:
        raise ValueError('Vo requires srad >= 1 and 0.1 <= step <= srad.')
    return opts


def find_vo_center(captured, rows=None, options=None):
    """Estimate COR from prepared sinograms, without candidate reconstructions."""
    assert_context(captured, check_source=True)
    require_compute_session()
    nproj, nrows, ncols = captured['dimensions']
    if rows is None:
        rows = sorted(set(int(round((nrows-1)*f)) for f in (.25, .5, .75)))
    else:
        rows = list(rows)
    if (len(rows) < 3 or len(set(rows)) != len(rows) or
            any(type(r) is not int or not 0 <= r < nrows for r in rows)):
        raise ValueError('Vo COR needs at least three distinct valid detector rows.')
    opts = vo_options(options)
    # Prevent native parallel jobs from exceeding a smaller explicit CPU assignment.
    available = max(1, os.cpu_count() or 1)
    if hasattr(os, 'sched_getaffinity'):
        try:
            available = min(available, max(1, len(os.sched_getaffinity(0))))
        except OSError:
            pass
    assigned = os.environ.get('SLURM_CPUS_PER_TASK', '')
    if assigned.isdecimal() and int(assigned) > 0:
        available = min(available, int(assigned))
    if opts['ncore'] > available:
        raise ValueError(f'Vo ncore={opts["ncore"]} exceeds the available CPU count {available}.')
    with h5py.File(captured['source']['path'], 'r') as f:
        theta = np.asarray(f['/exchange/theta'][:], dtype=np.float64)
    steps = np.diff(theta)
    if (theta.shape != (nproj,) or not np.isfinite(theta).all() or
            not np.all(steps > 0) or not np.allclose(steps, np.median(steps), rtol=.01, atol=1e-6) or
            abs(float(theta[-1]-theta[0])-180.) > max(1., 2*float(np.median(steps)))):
        raise ValueError('Vo COR requires approximately uniform angles spanning 180 degrees.')
    helper = load_als(captured['backend'])
    tomopy = helper.als.tomopy
    if not callable(getattr(tomopy, 'find_center_vo', None)):
        raise RuntimeError('The ALS kernel must provide tomopy.find_center_vo().')
    native = {name: opts[name] for name in ('smin','smax','srad','step','ratio','drop','ncore')}
    # Old TomoPy releases predate the ncore argument. Keep the native estimator usable,
    # while recording that those releases choose their own worker count.
    supports_ncore = 'ncore' in inspect.signature(tomopy.find_center_vo).parameters
    if not supports_ncore:
        native.pop('ncore')
    absolute, offsets, errors, diagnostics = [], [], [], []
    started = time.monotonic()
    for row in rows:
        try:
            tomo, angles = helper.als.read_data(
                captured['source']['path'], proj=None, sino=slice(row, row+1, 1),
                downsample_factor=1,
                preprocess_settings=copy.deepcopy(captured['settings']['preprocessing']),
                postprocess_settings=copy.deepcopy(captured['settings']['postprocessing']))
            if np.shape(tomo) != (nproj, 1, ncols):
                raise ValueError('Unexpected prepared sinogram dimensions.')
            np.testing.assert_allclose(angles, -np.deg2rad(theta), atol=1e-6)
            sino = np.asarray(tomo[:, 0, :], dtype=np.float32)
            if not np.isfinite(sino).all():
                raise ValueError('Nonfinite prepared sinogram.')
            low, high = np.percentile(sino, [1,99])
            if high-low <= max(float(np.max(np.abs(sino)))*1e-6, np.finfo(np.float32).eps):
                raise ValueError('Flat/blank sinogram; choose a row containing sample.')
            center = float(tomopy.find_center_vo(sino, **native))
            # The ALS wrapper expects an offset from ncols/2, not a detector index.
            offset = center - ncols/2.
            if not np.isfinite(center) or not 0 < center < ncols:
                raise ValueError('Vo returned an invalid detector center.')
            margin = min(opts['srad'], ncols/4.)
            lower = max(0., ncols//2 + opts['smin'])
            upper = min(ncols-1., ncols//2 + opts['smax'])
            if center <= lower+margin or center >= upper-margin:
                if 'near_vo_limit' not in diagnostics:
                    diagnostics.append('near_vo_limit')
            absolute.append(center)
            offsets.append(offset)
            errors.append(None)
            print(f'Vo row {row}: midpoint COR {offset:+.3f} px')
            del tomo, sino
        except (ValueError, AssertionError) as error:
            absolute.append(None)
            offsets.append(None)
            errors.append(f'{type(error).__name__}: {error}')
            print(f'Vo row {row}: review required ({error})')
    valid = [v for v in offsets if v is not None]
    center = float(np.median(valid)) if valid else None
    spread = float(np.ptp(valid)) if valid else None
    if any(error is not None for error in errors):
        diagnostics.append('invalid_or_blank_rows')
    if spread is not None and spread > opts['disagreement_pixels']:
        diagnostics.append('row_disagreement')
    assert_context(captured, check_source=True)
    return dict(center=center, rows=rows, row_centers=offsets, row_centers_absolute=absolute,
                row_errors=errors, row_spread=spread, options=opts,
                status='review_required' if diagnostics else 'auto_accepted',
                diagnostics=diagnostics, method='tomopy.find_center_vo',
                tomopy_version=getattr(tomopy, '__version__', 'unknown'),
                ncore_supported=supports_ncore, context=copy.deepcopy(captured),
                policy_version='vo_rows_v1', elapsed_seconds=time.monotonic()-started)


def checked_auto_center(result, captured):
    assert_context(captured)
    if not isinstance(result, dict) or result.get('context') != captured:
        raise ValueError('Automatic COR is missing or stale. Run Vo COR for the selected scan.')
    if (result.get('status') != 'auto_accepted' or result.get('diagnostics') or
            result.get('method') != 'tomopy.find_center_vo' or
            result.get('policy_version') != 'vo_rows_v1' or len(result.get('rows', [])) < 3):
        raise ValueError('COR review required: ' + ', '.join(result.get('diagnostics') or ['invalid Vo result']))
    offsets = result.get('row_centers', [])
    if (len(offsets) != len(result['rows']) or len(set(result['rows'])) != len(result['rows']) or
            any(v is None or not np.isfinite(v) for v in offsets) or
            any(result.get('row_errors', []))):
        raise ValueError('COR review required: incomplete Vo estimates.')
    opts = vo_options(result.get('options'))
    center = float(np.median(offsets))
    if np.ptp(offsets) > opts['disagreement_pixels'] or not np.isclose(center, result.get('center')):
        raise ValueError('COR review required: Vo row estimates disagree.')
    return center


def manual_start(captured, automatic, alignment):
    assert_context(captured)
    if isinstance(automatic, dict) and automatic.get('context') == captured and automatic.get('center') is not None:
        return float(automatic['center']), 'current automatic recommendation'
    if alignment is None:
        raise ValueError('Run projection alignment first; no usable recommendation exists.')
    return float(alignment), 'projection-alignment estimate (no usable automatic result)'


def cor_comparison(captured, offsets, rows, folder=None):
    assert_context(captured, check_source=True)
    cols = captured['dimensions'][2]
    if any(not np.isfinite(v) or abs(v) >= cols/2 for v in offsets):
        raise ValueError('Comparison offsets cross detector bounds.')
    if any(type(r) is not int or not 0 <= r < captured['dimensions'][1] for r in rows):
        raise ValueError('Comparison rows must be valid integer detector indices.')
    if folder:
        Path(folder).mkdir(parents=True, exist_ok=True)
    for row in rows:
        images = [als_chunk(captured['source']['path'], captured['settings'], float(v), row, row+1,
                            captured['backend'])[0] for v in offsets]
        lo, hi = image_display_limits(images[0])
        fig, axes = plt.subplots(1, len(images), figsize=(5*len(images),5), squeeze=False)
        for ax, value, image in zip(axes[0], offsets, images):
            ax.imshow(image, cmap='gray', vmin=lo, vmax=hi, interpolation='nearest')
            ax.set_title(f'Row {row}; midpoint COR {value:+.3f} px'); ax.axis('off')
            if folder:
                tifffile.imwrite(Path(folder)/f'row{row:05d}_center{value:+.5f}.tiff', image)
        if folder:
            fig.savefig(Path(folder)/f'comparison_row{row:05d}.png', dpi=150)
        plt.close(fig)
        zoom_image_viewer(images,
            titles=[f'Row {row}; midpoint COR {value:+.3f} px' for value in offsets],
            limits=[(lo,hi)]*len(images),captured=captured,title='Manual COR comparison')
    assert_context(captured, check_source=True)


def apply_manual(captured, center, rows, method, base):
    assert_context(captured, check_source=True)
    if not np.isfinite(center) or abs(center) >= captured['dimensions'][2]/2:
        raise ValueError('Invalid manual midpoint offset in detector pixels.')
    if not rows or len(set(rows)) != len(rows) or any(type(r) is not int or not 0 <= r < captured['dimensions'][1] for r in rows):
        raise ValueError('Manual review requires distinct valid detector rows.')
    selection = dict(center=float(center), rows=list(rows), method=method,
                     starting_center=float(base), adjustment_pixels=float(center-base),
                     context=copy.deepcopy(captured), applied_utc=utc_now())
    json_write(Path(captured['run_dir'])/'manual_cor.json', selection)
    globals()['MANUAL_COR_SELECTION'] = selection
    return selection



def automatic_cor_workflow(captured, info, rows=None, options=None):
    """Restore a frozen plan, or estimate/save a new COR with native Vo."""
    assert_context(captured, check_source=True)
    root = Path(captured['run_dir'])
    path = root/'run_plan.json'
    if path.exists():
        center, selection = resolve_center(captured, existing=read_plan(path))
        return freeze_run(captured, info, center, selection), None
    opts = vo_options(options)
    spec = dict(context=captured, rows=rows, options=opts, method='tomopy.find_center_vo')
    folder = root/'auto_cor_vo'/digest(spec)[:16]
    folder.mkdir(parents=True, exist_ok=True)
    json_write(folder/'settings.json', spec)
    with exclusive_lock(root/'.run.lock'), (folder/'vo.log').open('a', encoding='utf-8') as log:
        with contextlib.redirect_stdout(log), contextlib.redirect_stderr(log):
            result = find_vo_center(captured, rows, opts)
        json_write(folder/'result.json', result)
    if result['status'] != 'auto_accepted':
        return None, result
    center, selection = resolve_center(captured, automatic=result)
    return freeze_run(captured, info, center, selection), result


print('DONE: native Vo COR and manual review helpers loaded.')

In [ ]:
# 6. Define projection splitting helpers

def projection_view(source_path, destination, parity, expected_source):
    """Create and verify an even or odd HDF5 virtual projection view.

    The view stores angles and ALS metadata for the selected projections;
    image pixels and flat/dark frames remain in the unchanged source file.
    `parity` is 0 for even indices and 1 for odd indices.
    """
    source_path, destination = Path(source_path).resolve(), Path(destination)
    assert fingerprint(source_path) == expected_source, 'Source file moved or changed.'
    if parity not in (0, 1):
        raise ValueError('Projection parity must be 0 (even) or 1 (odd).')
    validate_als_metadata(source_path)
    nproj, rows, columns = expected_source['shape']
    indices = np.arange(parity, nproj, 2, dtype=np.int64)
    assert len(indices) >= 2
    signature = digest(dict(view_schema=4, source=expected_source, parity=parity))
    if not destination.exists():
        destination.parent.mkdir(parents=True, exist_ok=True)
        tmp = destination.with_name(destination.name + '.' + uuid.uuid4().hex + '.tmp')
        with h5py.File(source_path, 'r') as src, h5py.File(tmp, 'w', libver='latest') as out:
            original = src['/exchange/data']
            layout = h5py.VirtualLayout(shape=(len(indices), rows, columns), dtype=original.dtype)
            virtual = h5py.VirtualSource(str(source_path), '/exchange/data', shape=original.shape)
            for out_index, original_index in enumerate(indices):
                layout[out_index] = virtual[int(original_index)]
            ex = out.create_group('exchange')
            ex.create_virtual_dataset('data', layout)
            ex['data_white'] = h5py.ExternalLink(str(source_path), '/exchange/data_white')
            ex['data_dark'] = h5py.ExternalLink(str(source_path), '/exchange/data_dark')
            ex.create_dataset('theta', data=src['/exchange/theta'][indices])
            # ALS metadata for the modern read_metadata/read_data pipeline.
            values = {
                '/measurement/instrument/detector/dimension_x': columns,
                '/measurement/instrument/detector/dimension_y': rows,
                '/process/acquisition/rotation/num_angles': len(indices),
                '/process/acquisition/rotation/range': float(src['/exchange/theta'][int(indices[-1])] - src['/exchange/theta'][int(indices[0])]),
                '/process/acquisition/flat_fields/num_flat_fields': src['/exchange/data_white'].shape[0],
                '/process/acquisition/dark_fields/num_dark_fields': src['/exchange/data_dark'].shape[0],
                '/process/acquisition/flat_fields/i0cycle': 0,
            }
            for key, value in values.items():
                out.create_dataset(key, data=[value])
            for key in ('/measurement/instrument/detector/pixel_size',
                        '/measurement/instrument/camera_motor_stack/setup/camera_distance',
                        '/measurement/instrument/monochromator/energy'):
                if key in src:
                    out.create_dataset(key, data=src[key][()])
            out.create_dataset('source_projection_indices', data=indices)
            out.attrs['signature'] = signature
        tmp.replace(destination)
    with h5py.File(source_path, 'r') as src, h5py.File(destination, 'r') as view:
        assert view.attrs.get('signature') == signature, 'Stale view; choose a new run directory.'
        assert view['/exchange/data'].is_virtual
        assert view['/exchange/data'].shape == (len(indices), rows, columns)
        np.testing.assert_array_equal(view['source_projection_indices'][:], indices)
        np.testing.assert_array_equal(view['/exchange/theta'][:], src['/exchange/theta'][indices])
        for key in ('data_white', 'data_dark'):
            np.testing.assert_array_equal(view['/exchange/' + key][0], src['/exchange/' + key][0])
        for i in sorted(set([0, len(indices)//2, len(indices)-1])):
            np.testing.assert_array_equal(view['/exchange/data'][i], src['/exchange/data'][int(indices[i])])
    return destination


print("DONE: projection view helpers loaded.")

In [ ]:
# 7. Define reconstruction queue helpers

def validate_stack(folder, nrows, ncols):
    """Require every expected row TIFF to have the right name, shape, and type."""
    folder = Path(folder)
    files = tiff_paths(folder)
    upstream_files = sorted(folder.glob('*.tif*'))
    assert upstream_files == files, f'Unexpected files matched by Noise2Inverse (*.tif*): {folder}'
    expected_names = [f'recon_{i:05d}.tiff' for i in range(nrows)]
    assert [p.name for p in files] == expected_names, f'Incomplete or mixed TIFF stack: {folder}'
    for p in files:
        with tifffile.TiffFile(p) as tf:
            assert tf.series[0].shape == (ncols, ncols), f'Wrong shape: {p}'
            assert tf.series[0].dtype == np.dtype('float32'), f'Wrong dtype: {p}'
    return files


def prepare_views(saved):
    rec = validate_plan(saved)['reconstruction']
    folder = Path(saved['run_dir']) / 'projection_views'
    even = projection_view(rec['source']['path'], folder/'even.h5', 0, rec['source'])
    odd = projection_view(rec['source']['path'], folder/'odd.h5', 1, rec['source'])
    return even, odd


def run_plan(saved):
    rec = reconstruction_identity(saved)
    root = Path(saved['run_dir'])
    with exclusive_lock(root/'.run.lock'):
        if reconstruction_identity(read_plan(root/'run_plan.json')) != rec:
            raise ValueError('Run directory conflicts with queued reconstruction identity.')
        if fingerprint(rec['source']['path']) != rec['source']:
            raise ValueError('Source/signature mismatch; create a new v3 run.')
        load_als(rec['backend'])  # Exact saved source fingerprint, before any views or outputs.
        even, odd = prepare_views(saved)
        _, rows, cols = rec['dimensions']
        completed = 0
        for stage, input_path in (('full', rec['source']['path']), ('even', even), ('odd', odd)):
            output = safe_child(root, rec['outputs'][stage])
            signature = stage_signature(saved, stage)
            marker = root/(stage+'_complete.json')
            record = dict(schema=1, signature=signature, output_format='float32_row_tiff_v1')
            if marker.exists():
                old = json.loads(marker.read_text(encoding='utf-8'))
                if old.get('schema') != 1 or old.get('signature') != signature:
                    raise ValueError('Completion record format/signature mismatch; use a new run.')
                validate_stack(output, rows, cols)
                print('Already complete:', output)
                continue
            spec = root/(stage+'_command.json')
            if spec.exists():
                if json.loads(spec.read_text(encoding='utf-8')) != record:
                    raise ValueError('Tracked stage identity mismatch; use a new run.')
            else:
                if output.exists() and any(output.iterdir()):
                    raise ValueError(f'Untracked reconstruction output: {output}')
                json_write(spec, record)
            # Interrupted tracked stages rerun; per-image writes replace their own TIFFs.
            run_als_stage(input_path, output, rec['settings'], rec['center'], rows,
                          rec['backend'], root/'logs'/(stage+'.log'))
            validate_stack(output, rows, cols)
            json_write(marker, dict(record, completed_utc=utc_now()))
            completed += 1
            print('DONE: reconstruction stage', stage)
        return 'completed' if completed else 'already-complete'


def queue_lock(path):
    return Path(str(path)+'.lock')


def load_queue(path):
    path = Path(path)
    if not path.exists():
        return dict(schema=1, entries=[])
    try:
        queue = json.loads(path.read_text(encoding='utf-8'))
    except (ValueError, UnicodeError) as error:
        raise ValueError('Corrupt queue JSON. Use the optional queue repair cell; original file preserved.') from error
    if not isinstance(queue, dict) or queue.get('schema') != 1 or not isinstance(queue.get('entries'), list):
        raise ValueError('Unsupported queue format. Use queue repair to rebuild from selected v3 plans.')
    return queue


def inspect_queue(queue, expected_backend=None, expected_settings=None):
    """Cheap preflight: plan/backend checks; raw scans are opened only during execution."""
    results, ids, paths, roots = [], set(), set(), {}
    for number, entry in enumerate(queue['entries']):
        outcome = dict(index=number, id=entry.get('id') if isinstance(entry, dict) else None,
                       plan_path=entry.get('plan_path') if isinstance(entry,dict) else None,
                       status='ready', reason='')
        try:
            if (not isinstance(entry, dict) or not isinstance(entry.get('id'), str) or not entry['id'] or
                    not isinstance(entry.get('plan_path'), str) or not Path(entry['plan_path']).is_absolute()):
                raise ValueError('Malformed entry: expected stable ID and absolute schema-4 plan path.')
            path = str(Path(entry['plan_path']).resolve())
            if entry['id'] in ids:
                raise ValueError('Duplicate stable entry ID.')
            ids.add(entry['id'])
            if path in paths:
                outcome.update(status='skipped', reason='Duplicate plan path.')
            else:
                paths.add(path)
                saved = read_plan(path)
                rec = saved['reconstruction']
                key = str(Path(saved['run_dir']).resolve())
                signature = digest(rec)
                if key in roots and roots[key] != signature:
                    raise ValueError('Conflicting identities for one run directory.')
                roots[key] = signature
                if not Path(rec['source']['path']).is_file():
                    raise FileNotFoundError('Missing raw source: '+rec['source']['path'])
                if expected_backend is not None and rec['backend'] != expected_backend:
                    raise ValueError('Backend mismatch; configure the saved backend for this queue.')
                if expected_settings is not None and rec['settings'] != expected_settings:
                    raise ValueError('Settings mismatch; configure the saved settings for this queue.')
                directory = Path(rec['backend']['directory'])
                for name, sha in rec['backend']['source_sha256'].items():
                    if hashlib.sha256((directory/name).read_bytes()).hexdigest() != sha:
                        raise ValueError('Backend source fingerprint mismatch.')
                outcome['plan'] = saved
        except (KeyError, TypeError, ValueError, OSError, AssertionError) as error:
            outcome.update(status='blocked', reason=f'{type(error).__name__}: {error}')
        results.append(outcome)
    return results


def enqueue_plan(queue_file, plan_file):
    saved = read_plan(plan_file)
    path = str(Path(plan_file).resolve())
    with exclusive_lock(queue_lock(queue_file)):
        queue = load_queue(queue_file)
        if not any(isinstance(e, dict) and str(Path(e.get('plan_path','')).resolve()) == path for e in queue['entries']):
            queue['entries'].append(dict(id=uuid.uuid4().hex, plan_path=path))
            json_write(queue_file, queue)
    return queue


def execute_queue(queue_file, expected_backend=None, expected_settings=None, runner=None):
    runner = run_plan if runner is None else runner
    started = time.monotonic()
    outcomes = []
    report_path = Path(queue_file).with_name('queue_report_v3.json')
    with exclusive_lock(queue_lock(queue_file)):
        queue = load_queue(queue_file)
        preflight = inspect_queue(queue, expected_backend, expected_settings)
        try:
            for item in preflight:
                result = {k:v for k,v in item.items() if k != 'plan'}
                outcomes.append(result)
                if result['status'] == 'ready':
                    result['status'] = 'running'
                    try:
                        result['status'] = runner(item['plan'])
                        if result['status'] not in ('completed','already-complete'):
                            raise ValueError('Runner returned an unsupported outcome.')
                    except Exception as error:
                        result.update(status='failed', reason=f'{type(error).__name__}: {error}')
                print(f"Queue {result['id']}: {result['status']} {result['reason']}")
        except KeyboardInterrupt:
            if outcomes and outcomes[-1]['status'] == 'running':
                outcomes[-1].update(status='failed', reason='User interrupted this entry.')
            json_write(report_path, dict(schema=1, interrupted=True, outcomes=outcomes,
                                        elapsed_seconds=time.monotonic()-started, finished_utc=utc_now()))
            print('STOPPED: queue interrupted; later entries were not launched.')
            raise
        report = dict(schema=1, interrupted=False, outcomes=outcomes,
                      elapsed_seconds=time.monotonic()-started, finished_utc=utc_now())
        json_write(report_path, report)
    return report


def propose_queue_edit(queue_file, remove_ids=(), replacements=None, deduplicate=False,
                       remove_missing=False, rebuild=None):
    replacements = {} if replacements is None else replacements
    path = Path(queue_file)
    original = path.read_bytes() if path.exists() else None
    if rebuild is not None:
        queue = dict(schema=1, entries=[])
        for selected in rebuild:
            read_plan(selected)
            queue['entries'].append(dict(id=uuid.uuid4().hex, plan_path=str(Path(selected).resolve())))
    else:
        queue = copy.deepcopy(load_queue(path))
    existing_ids = {e.get('id') for e in queue['entries'] if isinstance(e, dict)}
    if not set(replacements).issubset(existing_ids) or not set(remove_ids).issubset(existing_ids):
        raise ValueError('Repair IDs must exist in the inspected queue.')
    entries, seen = [], set()
    for entry in queue['entries']:
        if not isinstance(entry, dict):
            entries.append(entry)
            continue
        if entry.get('id') in remove_ids:
            continue
        if entry.get('id') in replacements:
            replacement = replacements[entry['id']]
            read_plan(replacement)
            entry['plan_path'] = str(Path(replacement).resolve())
        p = entry.get('plan_path')
        if isinstance(p, str):
            p = str(Path(p).resolve())
            if remove_missing and not Path(p).is_file():
                continue
            if deduplicate and p in seen:
                continue
            seen.add(p)
        entries.append(entry)
    queue['entries'] = entries
    return dict(original_sha256=None if original is None else hashlib.sha256(original).hexdigest(),
                proposed=queue, preflight=inspect_queue(queue))


def apply_queue_edit(queue_file, proposal):
    path = Path(queue_file)
    with exclusive_lock(queue_lock(path)):
        original = path.read_bytes() if path.exists() else None
        sha = None if original is None else hashlib.sha256(original).hexdigest()
        if sha != proposal['original_sha256']:
            raise ValueError('Queue changed after inspection; preview the repair again.')
        refreshed = inspect_queue(proposal['proposed'])
        for old, new in zip(proposal['preflight'], refreshed):
            if old['status'] == 'ready' and (new['status'] != 'ready' or old.get('plan') != new.get('plan')):
                raise ValueError('Replacement plan changed after preview; inspect the repair again.')
        backup = path.with_name(path.name+'.backup.'+uuid.uuid4().hex)
        if original is not None:
            with backup.open('xb') as stream:
                stream.write(original)
        json_write(path, proposal['proposed'])
    return backup if original is not None else None


def remove_confirmed_stale_lock(path, original_job_stopped=False, expected_token=None):
    if not original_job_stopped or not expected_token:
        raise ValueError('Confirm the original job/kernel stopped and copy its exact lock token.')
    path = Path(path)
    if json.loads(path.read_text(encoding='utf-8')).get('token') != expected_token:
        raise ValueError('Lock changed; inspect its owner again.')
    path.unlink()

print("DONE: reconstruction queue and repair helpers loaded.")

In [ ]:
# 8. Define cleanup helpers

def cleanup_categories(saved):
    rec = validate_plan(saved)['reconstruction']
    stem = Path(rec['source']['path']).stem
    return dict(cor=['auto_cor','cor_preview','manual_cor_previews'],
        projection_views=['projection_views'], full=[rec['outputs']['full']],
        even=[rec['outputs']['even']], odd=[rec['outputs']['odd']],
        denoised=[stem+'_denoised_slices', stem+'_denoised_volume_2.5d', stem+'_denoised_volume_3d',
                  stem+'_denoised_slices_nafnet25d', stem+'_denoised_volume_2.5d_nafnet25d'],
        logs=['logs'], training=['TrainOutput', rec['outputs']['full']+'_config.yaml'])


def cleanup_boundary(saved, output_root):
    root_path, output_path = Path(saved['run_dir']), Path(output_root)
    root, output = root_path.resolve(), output_path.resolve()
    if root == output or not path_is_within(root, output) or not root.is_dir():
        raise ValueError('Cleanup requires a selected run strictly inside OUTPUT_ROOT.')
    for p in [root_path, *root_path.parents]:
        if is_link_like(p):
            raise ValueError('Cleanup rejects symlink/junction run ancestors.')
        if p == output_path:
            break
    protected = [Path(saved['reconstruction']['source']['path']).resolve(),
                 Path(saved['reconstruction']['backend']['directory']).resolve()]
    for name in ('ALS_MICROCT_DIR','INPUT_ROOT'):
        if name in globals():
            protected.append(Path(globals()[name]).resolve())
    for cmd in globals().get('DENOISE_CMD', []):
        if Path(cmd).is_absolute():
            protected.append(Path(cmd).resolve().parent)
    if any(path_is_within(p, root) for p in protected):
        raise ValueError('Run contains raw inputs or backend/environment paths; cleanup rejected.')
    return root


def cleanup_manifest(saved, output_root, categories):
    root = cleanup_boundary(saved, output_root)
    if (root/'.run.lock').exists():
        raise RuntimeError('Selected run is active or locked. Cleanup rejected.')
    if reconstruction_identity(read_plan(root/'run_plan.json')) != reconstruction_identity(saved):
        raise ValueError('Selected cleanup plan differs from the frozen run.')
    available = cleanup_categories(saved)
    if len(set(categories)) != len(categories) or not set(categories).issubset(available):
        raise ValueError('Select named generated-output categories only; no paths or traversal.')
    targets, files = [], []
    def walk(path):
        if is_link_like(path):
            raise ValueError(f'Cleanup rejects links/junctions: {path}')
        resolved = path.resolve()
        if resolved == root or not path_is_within(resolved, root):
            raise ValueError(f'Cleanup target escapes run: {path}')
        stat = path.stat()
        item = dict(path=str(resolved), size=stat.st_size, mtime_ns=stat.st_mtime_ns,
                    inode=stat.st_ino, directory=path.is_dir())
        files.append(item)
        if path.is_dir():
            for child in sorted(path.iterdir()):
                walk(child)
    for category in categories:
        for name in available[category]:
            path = safe_child(root, name)
            if path.exists() or path.is_symlink():
                targets.append(dict(category=category, path=str(path.resolve())))
                walk(path)
    effects = []
    if set(categories) & {'even','odd'}:
        effects.append('Paired-stack deletion prevents training until reconstruction is rerun.')
    if 'full' in categories:
        effects.append('Full-stack deletion prevents reconstruction previews and inference that need it.')
    if 'training' in categories:
        effects.append('Deleting models removes resume/checkpoint capability and saved normalization.')
    return dict(run_dir=str(root), output_root=str(Path(output_root).resolve()),
        reconstruction_sha256=digest(reconstruction_identity(saved)), categories=list(categories),
        targets=targets, files=files, file_count=sum(not f['directory'] for f in files),
        estimated_bytes=sum(f['size'] for f in files if not f['directory']), effects=effects)


def execute_cleanup(saved, output_root, preview, dry_run=True, confirmation='', opt_in=False):
    if dry_run:
        return dict(status='dry-run', preview=preview, deleted=[], failed=[], estimated_reclaimed_bytes=0)
    root = cleanup_boundary(saved, output_root)
    if not opt_in or confirmation != str(root):
        raise ValueError('Cleanup requires explicit opt-in and the exact resolved RUN_DIR confirmation.')
    if (root/'.run.lock').exists():
        raise RuntimeError('Selected run is active or locked.')
    current = cleanup_manifest(saved, output_root, preview['categories'])
    if current != preview:
        raise ValueError('Cleanup selection changed since preview. Run the dry-run cell again.')
    deleted, failed, reclaimed = [], [], 0
    with exclusive_lock(root/'.run.lock'):
        # Recheck all paths immediately before deletion while holding the run lock.
        for item in preview['files']:
            path = Path(item['path'])
            if is_link_like(path) or path.resolve() != path:
                raise ValueError('Cleanup path changed into a link; preview again.')
            stat = path.stat()
            if (stat.st_size, stat.st_mtime_ns, stat.st_ino) != (item['size'], item['mtime_ns'], item['inode']):
                raise ValueError('Cleanup files changed; preview again.')
        affected = {t['category'] for t in preview['targets']} & {'full','even','odd'}
        # Mark stages incomplete before deletion, including partially failed deletions.
        for stage in affected:
            (root/(stage+'_complete.json')).unlink(missing_ok=True)
        # Delete only reviewed paths, deepest first; never recursively follow fresh contents.
        for item in sorted(preview['files'], key=lambda f: len(Path(f['path']).parts), reverse=True):
            path = Path(item['path'])
            try:
                if is_link_like(path) or path.resolve() != path or not path_is_within(path, root):
                    raise OSError('Cleanup target changed or escaped its reviewed boundary.')
                if item['directory']:
                    path.rmdir()
                else:
                    path.unlink()
                    reclaimed += item['size']
                deleted.append(str(path))
            except OSError as error:
                failed.append(dict(path=str(path), reason=f'{type(error).__name__}: {error}'))
        report = dict(status='partial-failure' if failed else 'completed', deleted=deleted,
            failed=failed, estimated_reclaimed_bytes=reclaimed, preview=preview, finished_utc=utc_now())
        json_write(root/('cleanup_report_'+uuid.uuid4().hex+'.json'), report)
    if affected or set(preview['categories']) & {'training','denoised'}:
        for name in ('PAIR_CONTEXT','TRAINING_CONTEXT','stack_lists','CONFIG_YAML','TRAIN_DIR','VOLUME_DIR'):
            globals().pop(name, None)
    return report

print("DONE: scoped cleanup helpers loaded.")

In [ ]:
# 9. Define training, inference, and history helpers

def validate_denoise_config(cfg, run_dir, full_dir, even_dir, odd_dir, ncols, *, nrows=None, normalized=False, volume=False):
    """Check dataset identity and mode-specific settings before launching the CLI."""
    if not isinstance(cfg, dict) or not all(isinstance(cfg.get(k), dict) for k in ('dataset', 'train', 'infer')):
        raise ValueError('The YAML must contain dataset, train, and infer mappings.')
    expected = dict(directory_to_reconstructions=str(run_dir), full_recon_name=full_dir.name,
                    sub_recon_name0=even_dir.name, sub_recon_name1=odd_dir.name)
    for key, value in expected.items():
        if cfg['dataset'].get(key) != value:
            raise ValueError(f'Wrong configuration path: {key}. Rerun cell 21 for the selected run.')
    train = cfg['train']
    mode = train.get('mode', '2.5d')
    if mode not in ('2.5d', '3d'):
        raise ValueError('Training mode must be 2.5d or 3d.')
    for key in ('psz', 'n_slices', 'mbsz', 'maxep'):
        if type(train.get(key)) is not int or train[key] <= 0:
            raise ValueError(f'train.{key} must be a positive integer.')
    if train.get('z_stride', 1) != 1:
        raise ValueError('This full-resolution workflow requires z_stride=1.')
    if mode == '2.5d':
        if train['psz'] >= ncols or train['psz'] % 8:
            raise ValueError('psz must be smaller than the image width and divisible by 8.')
        if train['n_slices'] % 2 == 0:
            raise ValueError('n_slices must be odd.')
    else:
        for key in ('psz_3d', 'nb_patches_3d', 'n_blocks_3d', 'start_filts_3d'):
            if type(train.get(key)) is not int or train[key] <= 0:
                raise ValueError(f'train.{key} must be a positive integer.')
        patch = train['psz_3d']
        if nrows is None or patch > min(nrows, ncols):
            raise ValueError('psz_3d must fit the volume depth and image width; provide nrows.')
        if train['n_blocks_3d'] > patch.bit_length() or patch % (2 ** train['n_blocks_3d']):
            raise ValueError('psz_3d must be divisible by 2**n_blocks_3d, following the upstream model contract.')
    architecture = train.get('architecture', 'unet3d' if mode == '3d' else 'unet')
    if architecture not in (('unet', 'nafnet25d') if mode == '2.5d' else ('unet3d',)):
        raise ValueError('Architecture is incompatible with the selected mode.')
    if architecture == 'nafnet25d' and train['n_slices'] < 3:
        raise ValueError('NAFNet requires an odd slice count >= 3.')
    if train.get('precision', 'fp32') not in ('fp32', 'bf16'):
        raise ValueError('precision must be fp32 or bf16.')
    for key in ('warmup', 'patience'):
        if type(train.get(key, 0)) is not int or train.get(key, 0) < 0:
            raise ValueError(f'train.{key} must be a nonnegative integer.')
    if not isinstance(train.get('lr'), (int, float)) or not np.isfinite(train['lr']) or train['lr'] <= 0:
        raise ValueError('train.lr must be finite and positive.')
    overlap = cfg['infer'].get('overlap')
    if not isinstance(overlap, (int, float)) or not np.isfinite(overlap) or not 0 <= overlap < 1:
        raise ValueError('infer.overlap must be in [0, 1).')
    if cfg['infer'].get('window') not in ('hann', 'cosine', 'uniform'):
        raise ValueError('infer.window must be hann, cosine, or uniform.')
    if normalized:
        for key in ('mean4norm', 'std4norm'):
            value = cfg['dataset'].get(key)
            if not isinstance(value, (int, float)) or not np.isfinite(value):
                raise ValueError(f'Missing or invalid training normalization: {key}. Restore the training YAML.')
        if cfg['dataset']['std4norm'] <= 0:
            raise ValueError('Training normalization standard deviation must be positive.')
    return cfg

def validate_checkpoint(train_dir, checkpoint):
    """Resolve a supported checkpoint without starting inference or creating outputs."""
    names = {'lcl': 'best_lcl_model.pth', 'val': 'best_val_model.pth', 'edge': 'best_edge_model.pth'}
    if checkpoint not in names:
        raise ValueError('CHECKPOINT must be lcl, val, or edge.')
    path = Path(train_dir) / names[checkpoint]
    if not path.is_file():
        raise FileNotFoundError(f'Requested checkpoint is not saved yet: {path}')
    return path

def validate_denoised_volume(folder, nrows, ncols):
    """Check output names, shape, floating dtype, and finite pixels one slice at a time."""
    paths = tiff_paths(folder)
    if [p.name for p in paths] != [f'{i:05d}.tiff' for i in range(nrows)]:
        raise ValueError('Denoised volume is incomplete or contains unexpected TIFFs; inspect its log.')
    for path in paths:
        image = tifffile.imread(path)
        if image.shape != (ncols, ncols) or image.dtype.kind != 'f' or not np.isfinite(image).all():
            raise ValueError(f'Invalid denoised image: {path}')
    return paths

print("DONE: training and inference validation helpers loaded.")

def parse_training_history(log_path):
    """Read epoch means; retain the latest complete record for resumed epochs."""
    path = Path(log_path)
    metadata = dict(mode=None, gpu_count=None, stop_reason=None, best_l1=None, best_lcl=None,
                    incomplete_epochs=[], duplicate_epochs=[])
    if not path.is_file():
        return dict(rows=[], metadata=metadata, log_path=str(path))
    number = r'[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?'
    metrics = re.compile(r'\[(Train|Val)\]\s+(L1 loss|LCL loss|EDGE Value|L2 loss|SSIM loss):\s*('+number+r')')
    names = {('Train','L1 loss'):'train_l1', ('Val','L1 loss'):'eval_l1',
             ('Train','LCL loss'):'train_lcl', ('Val','LCL loss'):'eval_lcl',
             ('Val','EDGE Value'):'edge',
             ('Train','L2 loss'):'train_l2', ('Val','L2 loss'):'eval_l2',
             ('Train','SSIM loss'):'train_ssim', ('Val','SSIM loss'):'eval_ssim'}
    required = ('train_l1', 'eval_l1', 'train_lcl', 'eval_lcl')
    records = {}
    current = None

    def commit():
        if current is None:
            return
        if all(key in current and np.isfinite(current[key]) for key in required):
            if current['epoch'] in records:
                metadata['duplicate_epochs'].append(current['epoch'])
            row = dict(current)
            row.setdefault('warmup', None)
            row.setdefault('edge', None)
            for key in ('train_l2','eval_l2','train_ssim','eval_ssim'):
                row.setdefault(key, None)
            row['ssim_data_range'] = metadata.get('ssim_data_range')
            records[row['epoch']] = row
        else:
            metadata['incomplete_epochs'].append(current['epoch'])

    with path.open('r', encoding='utf-8', errors='replace') as stream:
        for raw in stream:
            line = re.sub(r'\x1b\[[0-?]*[ -/]*[@-~]', '', raw).rstrip()
            command = re.search(r'\bdenoise\s+train\b', line)
            fresh = command and '--resume' not in line
            if fresh or 'Initializing model from scratch' in line or 'Fine-tuning from:' in line:
                # A new fresh/finetune invocation starts a new history, while resume appends.
                records.clear()
                current = None
                metadata.update(best_l1=None, best_lcl=None, stop_reason=None,
                                incomplete_epochs=[], duplicate_epochs=[])
            elif command:
                commit()
                current = None
                metadata['stop_reason'] = None
            metric_range = re.search(r'ssim_data_range=('+number+r')', line)
            if metric_range:
                metadata['ssim_data_range'] = float(metric_range[1])
            mode = re.search(r'(?:Training mode:\s*|--mode\s+)(2\.5d|3d)\b', line)
            if mode:
                metadata['mode'] = mode[1]
            gpu_count = re.search(r'Launching training via torchrun \((\d+) GPU', line)
            if gpu_count:
                metadata['gpu_count'] = int(gpu_count[1])
            elif command:
                assignment = re.search(r'--gpus\s+(\S+)', line)
                if assignment:
                    metadata['gpu_count'] = len(assignment[1].split(','))
            epoch = re.search(r'\bEpoch\s+(\d+)\s*$', line)
            if epoch:
                commit()
                current = dict(epoch=int(epoch[1]))
                continue
            if current is not None:
                metric = metrics.search(line)
                if metric:
                    current[names[(metric[1],metric[2])]] = float(metric[3])
                warmup = re.search(r'Is model warming up\?:\s*(True|False)', line)
                if warmup:
                    current['warmup'] = warmup[1] == 'True'
            for key, label in (('best_l1','Lowest model validation loss'),
                               ('best_lcl','Lowest model LCL loss')):
                best = re.search(label+r'\s+('+number+r')\s+at epoch\s+(\d+)', line)
                if best and int(best[2]) > 0:
                    metadata[key] = dict(value=float(best[1]), epoch=int(best[2]))
            if 'Early stopping:' in line:
                metadata['stop_reason'] = 'early stopping'
    commit()
    rows = [records[key] for key in sorted(records)]
    metadata['warmup_end'] = next((row['epoch'] for row in rows if row['warmup'] is False), None) \
        if any(row['warmup'] is True for row in rows) else None
    return dict(rows=rows, metadata=metadata, log_path=str(path))


def plot_training_history(log_path, output_dir=None, moving_window=10, show=True, outcome=None):
    """Save raw/smoothed L1 and LCL plots without loading a model or checkpoint."""
    if type(moving_window) is not int or moving_window < 1:
        raise ValueError('moving_window must be a positive integer.')
    history = parse_training_history(log_path)
    rows, metadata = history['rows'], history['metadata']
    if not rows:
        print('READY: no complete training-loss epochs yet in', log_path)
        return dict(status='no_history', log_path=str(log_path), epochs=0)
    log_path = Path(log_path)
    folder = Path(output_dir) if output_dir is not None else log_path.parent.parent/'training_history'
    folder.mkdir(parents=True, exist_ok=True)
    epochs = np.arange(rows[0]['epoch'], rows[-1]['epoch']+1)
    values = {key:np.full(len(epochs), np.nan) for key in ('train_l1','eval_l1','train_lcl','eval_lcl','train_l2','eval_l2','train_ssim','eval_ssim')}
    for row in rows:
        for key in values:
            values[key][row['epoch']-epochs[0]] = row[key]

    def smooth(array):
        result = np.full(len(array), np.nan)
        if len(array) >= moving_window:
            result[moving_window-1:] = np.convolve(array, np.ones(moving_window)/moving_window, mode='valid')
        return result

    def best_record(key, metric):
        tracked = metadata[key]
        if tracked is not None:
            return tracked
        candidates = rows
        if key == 'best_lcl' and metadata['warmup_end'] is not None:
            candidates = [row for row in rows if row['epoch'] > metadata['warmup_end']]
        if not candidates:
            return None
        row = min(candidates, key=lambda item:item[metric])
        return dict(epoch=row['epoch'], value=row[metric])

    best_l1 = best_record('best_l1','eval_l1')
    best_lcl = best_record('best_lcl','eval_lcl')
    paths = {name:folder/('training_loss_history.'+name) for name in ('png','svg')}
    paths['csv'] = folder/'training_history.csv'
    temp_csv = folder/('.training_history.'+uuid.uuid4().hex+'.tmp')
    try:
        with temp_csv.open('w', newline='', encoding='utf-8') as stream:
            writer = csv.DictWriter(stream, fieldnames=['epoch','train_l1','eval_l1','train_lcl','eval_lcl','edge','warmup',
                'train_l2','eval_l2','train_ssim','eval_ssim','ssim_data_range'])
            writer.writeheader()
            writer.writerows(rows)
        temp_csv.replace(paths['csv'])
    finally:
        temp_csv.unlink(missing_ok=True)

    style = {'font.family':'DejaVu Sans','font.size':11,'axes.labelcolor':'#243447',
             'text.color':'#243447','axes.edgecolor':'#cad4de','xtick.color':'#556575',
             'ytick.color':'#556575','axes.spines.top':False,'axes.spines.right':False,
             'svg.fonttype':'none'}
    with plt.rc_context(style):
        has_reference = any(np.isfinite(values[key]).any() for key in ('train_l2','eval_l2','train_ssim','eval_ssim'))
        fig, axes = plt.subplots(4 if has_reference else 2,1,figsize=(12,15 if has_reference else 8.8),sharex=True)
        try:
            fig.patch.set_facecolor('white')
            mode = metadata['mode'] or 'CT'
            mode = '2.5D' if mode == '2.5d' else ('3D' if mode == '3d' else mode)
            fig.suptitle(f'Noise2Inverse360 | {mode} training loss history',fontsize=20,
                         fontweight='bold',x=.08,ha='left',y=.978)
            subtitle = f'{len(rows)} epochs'
            if metadata['gpu_count'] is not None:
                subtitle += f' · {metadata["gpu_count"]} GPUs'
            reason = metadata['stop_reason'] or outcome
            if reason:
                subtitle += f' · {reason} after epoch {rows[-1]["epoch"]}'
            fig.text(.08,.936,subtitle,fontsize=12,color='#596b7d')
            colors = {'Training':'#2166ac', 'Evaluation':'#d26026'}
            specifications = [(axes[0],'train_l1','eval_l1','L1 loss','Epoch-average L1 loss',best_l1),
                              (axes[1],'train_lcl','eval_lcl','Local contrast loss (LCL)',
                               'Epoch-average weighted LCL loss',best_lcl)]
            if has_reference:
                specifications.extend([(axes[2],'train_l2','eval_l2','L2 / MSE (reference only)',
                    'Epoch-average paired MSE',None),
                    (axes[3],'train_ssim','eval_ssim','SSIM loss (reference only)',
                    'Epoch-average paired 1 - SSIM',None)])
            for ax, train_key, eval_key, title, ylabel, best in specifications:
                ax.set_facecolor('#fbfcfe')
                end = metadata['warmup_end']
                if end is not None:
                    ax.axvspan(epochs[0],end,color='#ccd5df',alpha=.25,zorder=0)
                    ax.axvline(end,color='#778798',linewidth=1,linestyle='--',zorder=1)
                for label, key in (('Training',train_key),('Evaluation',eval_key)):
                    averaged = smooth(values[key])
                    has_average = np.isfinite(averaged).any()
                    ax.plot(epochs,values[key],color=colors[label],alpha=.22 if has_average else .8,
                            linewidth=.8,marker='.' if len(rows) == 1 else None,
                            label=None if has_average else label,zorder=2)
                    if has_average:
                        ax.plot(epochs,averaged,color=colors[label],linewidth=2.2,label=label,zorder=3)
                ax.set_title(title,loc='left',fontweight='bold',fontsize=14,pad=12)
                ax.set_ylabel(ylabel)
                ax.grid(axis='y',color='#dce3ea',linewidth=.7)
                ax.set_xlim(epochs[0]-.5,epochs[-1]+.5)
                ax.tick_params(axis='both',length=0,pad=7)
                ax.legend(loc='upper right',frameon=False,ncol=2)
                if best is not None:
                    label = 'Best evaluation L1' if train_key == 'train_l1' else 'Best tracked evaluation LCL'
                    note = f'{label}: {best["value"]:.6f}\nEpoch {best["epoch"]}'
                    position = (.64,.8) if train_key == 'train_l1' else (.14,.72)
                    box = dict(boxstyle='round,pad=.5',facecolor='white',edgecolor='#e4e8ed')
                    if epochs[0] <= best['epoch'] <= epochs[-1]:
                        ax.scatter(best['epoch'],best['value'],marker='*',s=155,
                                   color=colors['Evaluation'],edgecolors='white',linewidth=.8,zorder=5)
                        ax.annotate(note,xy=(best['epoch'],best['value']),xytext=position,
                                    textcoords='axes fraction',fontsize=10,bbox=box,
                                    arrowprops=dict(arrowstyle='-',color='#a7b2be',linewidth=1))
                    else:
                        ax.text(*position,note+' (earlier history)',transform=ax.transAxes,fontsize=10,bbox=box)
            if metadata['warmup_end'] is not None:
                axes[0].text(metadata['warmup_end']+5,.88,'LCL warmup ended',
                             transform=axes[0].get_xaxis_transform(),fontsize=9,color='#687889')
            axes[-1].set_xlabel('Epoch',labelpad=10)
            fig.text(.08,.047,f'Faint lines: raw epoch averages. Solid lines: trailing {moving_window}-epoch moving averages.',
                     fontsize=10,color='#596b7d')
            fig.text(.08,.021,'Evaluation uses the training dataset; logged metrics represent one GPU process. LCL is plotted with its logged weight. L2/SSIM are reference metrics only.',
                     fontsize=9,color='#596b7d')
            fig.subplots_adjust(left=.105,right=.97,top=.87,bottom=.12,hspace=.31)
            for extension in ('png','svg'):
                temp = folder/('.training_loss_history.'+uuid.uuid4().hex+'.tmp')
                try:
                    fig.savefig(temp,format=extension,dpi=180,facecolor='white')
                    temp.replace(paths[extension])
                finally:
                    temp.unlink(missing_ok=True)
            if show:
                plt.show()
        finally:
            plt.close(fig)
    print('DONE: saved training history:', paths['png'])
    return dict(status='saved',epochs=len(rows),first_epoch=int(epochs[0]),last_epoch=int(epochs[-1]),
                best_l1=best_l1,best_lcl=best_lcl,metadata=metadata,
                paths={key:str(value) for key,value in paths.items()})

## Select paths and a scan: cells 10–11

Edit cell 10's input folder and preferred filename before running it. Its examples
are `$PSCRATCH/raw_microct/sample.h5`; set the scan filename and ensure the directory
is mounted on the compute node. `RUN_LABEL` names this scan's particular settings
and model run. Use a new label for changed reconstruction/COR or training settings.

`ALS_MICROCT_DIR` points to the **source checkout**, not its Conda prefix.
`DENOISE_CMD` points to the **n2i executable**, not the repository folder. The setup
above supplies both defaults. `OUTPUT_ROOT_OVERRIDE=None` selects scratch from
the environment; set an absolute `Path` if the service uses another location.

Select the intended scan in the dropdown (or set `default_scan` if widgets are
unavailable), then run cell 11 to inspect dimensions, angles and metadata.
Running these cells alone does not reconstruct or train anything.


In [ ]:
# 10. Select paths and scan

# USER INPUT: REQUIRED paths/scan/run label; OPTIONAL scratch; ADVANCED backend/GPU.
try:
    INPUT_SUBFOLDER = 'raw_microct'       # Raw scan directory under scratch
    INPUT_ROOT = Path(os.environ.get('PSCRATCH') or os.environ.get('SCRATCH') or '/alsdata') / INPUT_SUBFOLDER
    # Alternative: INPUT_ROOT = Path('/absolute/path/to/raw/scans').
    OUTPUT_ROOT_OVERRIDE = None            # Default: $PSCRATCH/$SCRATCH; optional explicit scratch Path
    RUN_LABEL = 'v3_3_nafnet_1'                # One label per source/center/settings combination
    PREFERRED_FILE = 'sample.h5'  # Raw scan filename
    ALS_MICROCT_DIR = Path.home() / 'microct-n2i-source' / 'microct'  # Home source checkout
    ALS_USE_GPU = None            # None: upstream detection; False: CPU; True: allocated GPU
    DENOISE_CMD = [str(Path.home() / 'microct-n2i-envs' / 'n2i' / 'bin' / 'denoise')]
    
    # Inference uses one GPU; cell 21 selects every allocated GPU for training.
    INFERENCE_GPU_ID = first_visible_gpu()  # Preserve an explicitly disabled mask
    
    invalidate_downstream()
    globals()['WORKFLOW_CONTEXT'] = None
    OUTPUT_ROOT = select_output_root(OUTPUT_ROOT_OVERRIDE).resolve()
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    assert INPUT_ROOT.is_dir(), f'Input folder not found: {INPUT_ROOT}'
    scan_files = sorted(p for p in INPUT_ROOT.iterdir() if p.is_file() and p.suffix.lower() in ('.h5', '.hdf5'))
    assert scan_files, f'No HDF5 files in {INPUT_ROOT}'
    default_scan = next((p for p in scan_files if p.name == PREFERRED_FILE), scan_files[0])
    if widgets:
        file_picker = widgets.Dropdown(options=[(p.name, str(p)) for p in scan_files],
                                      value=str(default_scan), description='Scan:',
                                      layout=widgets.Layout(width='95%'))
        display(file_picker)
    else:
        file_picker = None
        print('\n'.join(p.name for p in scan_files))
        print('Set default_scan to the desired Path before continuing.')
    print('Output:', OUTPUT_ROOT)
    print('Filesystem free space (not your remaining quota):', round(shutil.disk_usage(OUTPUT_ROOT).free / 2**30, 1), 'GiB')
    
    if file_picker is not None:
        def selection_changed(change):
            invalidate_downstream()
            globals()['WORKFLOW_CONTEXT'] = None
            print('READY: scan selection changed; inspect and configure the selected scan.')
        file_picker.observe(selection_changed, names='value')
    print('READY: paths and scan selector initialized; inspect the selected scan next.')
except Exception:
    print("FAILED: paths; see exception below.")
    raise

In [ ]:
# 11. Inspect the selected scan

# USER INPUT: REQUIRED select scan above; REQUIRED new RUN_LABEL for changed identity.
try:
    RAW_H5 = Path(file_picker.value if file_picker else default_scan).resolve()
    if RUN_LABEL in ('.', '..') or not re.fullmatch(r'[A-Za-z0-9_.-]+', RUN_LABEL):
        raise ValueError('Use a run label with letters, digits, underscores, hyphens, or dots; not the special directory names dot or dot-dot.')
    RUN_DIR = OUTPUT_ROOT / RAW_H5.stem / RUN_LABEL
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    INFO = inspect_scan(RAW_H5)
    SOURCE = fingerprint(RAW_H5)
    NPROJ, NROWS, NCOLS = INFO['shape']
    print(json.dumps(INFO, indent=2))
    if INFO['energy_kev'] is None:
        print('Energy metadata is invalid/missing. Obtain the actual spectrum/energy before phase retrieval.')
    if NCOLS % 2:
        raise ValueError('Odd detector width: choose an explicit crop before using this notebook.')
    stack_gib = NROWS * NCOLS * NCOLS * 4 / 2**30
    print(f'Float32 TIFF stack: {stack_gib:.2f} GiB; full + even + odd: {3 * stack_gib:.2f} GiB')
    print(f'Two training arrays alone: {2 * stack_gib:.2f} GiB system RAM per training process.')
    PLAN_FILE = RUN_DIR / 'run_plan.json'
    if PLAN_FILE.exists():
        saved = read_plan(PLAN_FILE)
        if saved['reconstruction']['source'] != SOURCE:
            raise ValueError('Frozen source changed: use a new v3 RUN_LABEL.')
    elif (RUN_DIR / 'v3_run_identity.json').exists():
        identity = json.loads((RUN_DIR / 'v3_run_identity.json').read_text(encoding='utf-8'))
        if identity.get('notebook_version') not in ('3.0', '3.1', '3.2', '3.3') or identity.get('source') != SOURCE:
            raise ValueError('Run identity changed: use a new v3 RUN_LABEL.')
    elif any(RUN_DIR.iterdir()):
        raise ValueError('Existing unplanned directory: use a new v3 RUN_LABEL.')
    else:
        json_write(RUN_DIR / 'v3_run_identity.json', dict(notebook_version='3.3', source=SOURCE))
    select_context(SOURCE, RUN_DIR, INFO)
    json_write(RUN_DIR / 'scan_metadata.json', INFO)
    print('DONE: scan inspected; configure reconstruction next.')
except Exception:
    print("FAILED: inspect-selected; see exception below.")
    raise

## Configure reconstruction: cell 12

The notebook calls the ALS backend's `helper.reconstruct` function using the settings below; the upstream ALS notebook is not required. `method='fbp'` selects ASTRA FBP (CPU or GPU); `gridrec` selects CPU TomoPy gridrec. The explicit method keeps queued runs consistent across machines. `fc` is the backend's low-pass cutoff; start at 0.75 and compare previews. Reconstruction uses full detector resolution, circular masking, and upstream attenuation scaling.

`preprocessing` operates **after flat/dark normalization, before minus-log**: the outlier threshold is in transmission units, not the legacy raw-count threshold of 750. Zero disables outlier removal. `sm_size=0` disables the upstream SAREPY-style stripe routine; enabling it also requires `snr` and `la_size`. `postprocessing.ringSigma=0` disables Fourier-wavelet stripe removal; a positive value enables it with `ringLevel`. Phase retrieval and exposure-ratio correction are not exposed by this backend wrapper; this notebook assumes equal flat/projection exposure.

Start with corrections off, then compare their effect on previews. Use the same settings for full/even/odd. `als_chunk_sino` limits rows loaded per call; 25 is a conservative starting point. No projection downsampling or 360-to-180 conversion is applied.

| Setting | Interpretation |
|---|---|
| `method` | `fbp`: ASTRA FBP; `gridrec`: TomoPy gridrec. These can produce different filtering and image appearance. |
| `fc` | Cutoff in `(0, 1]`; smaller values smooth more. FBP and gridrec implement filtering differently, so the same number does not imply identical results. |
| `preprocessing.minimum_transmission` | Positive floor on normalized transmission before minus-log; prevents taking log of zero/negative values. |
| `preprocessing.outlier_diff_1D`, `outlier_size_1D` | Transmission difference threshold and window size for filtering along the angle dimension; threshold 0 disables it. |
| `preprocessing.sm_size`, `la_size`, `snr` | Small/large stripe windows and sensitivity for upstream stripe removal; `sm_size=0` disables it. Use suitable odd window sizes when enabled. |
| `postprocessing.ringSigma`, `ringLevel` | Fourier-wavelet damping and decomposition level; `ringSigma=0` disables it. |
| `als_chunk_sino` | Number of detector rows loaded per reconstruction call. It does not limit Noise2Inverse's memory use. |

Inspect changes before freezing the plan. Even a chunk-size or CPU/GPU change changes this notebook's saved settings and requires a new run label. Full/even/odd must be regenerated together when image-forming settings change.

In [ ]:
# 12. Set reconstruction options

# USER INPUT: ADVANCED reconstruction processing settings below.
try:
    RECON_SETTINGS = {
        'als_chunk_sino': 25,
        'method': 'fbp',
        'fc': 0.75,
        'preprocessing': {'minimum_transmission': 0.01,
                          'outlier_diff_1D': 0.0, 'outlier_size_1D': 3,
                          'snr': 3.0, 'la_size': 51, 'sm_size': 0},
        'postprocessing': {'ringSigma': 0.0, 'ringLevel': 8},
    }
    assert RECON_SETTINGS['method'] in ('fbp', 'gridrec')
    assert RECON_SETTINGS['als_chunk_sino'] > 0
    assert 0 < RECON_SETTINGS['fc'] <= 1
    assert 0 < RECON_SETTINGS['preprocessing']['minimum_transmission'] <= 1
    validate_als_metadata(RAW_H5)
    
    # Load exactly this checkout, including helper's sibling import.
    ALS_BACKEND_DIR = ALS_MICROCT_DIR.resolve() / 'backend'
    ALS_FILES = ('ALS_recon_functions.py', 'ALS_recon_helper.py')
    assert all((ALS_BACKEND_DIR / name).is_file() for name in ALS_FILES), \
        'Set ALS_MICROCT_DIR to the microct checkout containing backend/ALS_recon_helper.py.'
    BACKEND_RECORD = dict(metadata_adapter='optional_camera_distance_v1', name='als_modern', directory=str(ALS_BACKEND_DIR),
        source_sha256={name: hashlib.sha256((ALS_BACKEND_DIR / name).read_bytes()).hexdigest()
                       for name in ALS_FILES})
    
    
    require_compute_session()
    ALS_HELPER = load_als(BACKEND_RECORD)
    RECON_SETTINGS['use_gpu'] = bool(ALS_HELPER.als.check_for_gpu() if ALS_USE_GPU is None else ALS_USE_GPU)
    if RECON_SETTINGS['method'] == 'gridrec':
        RECON_SETTINGS['use_gpu'] = False
    if RECON_SETTINGS['use_gpu']:
        validate_gpu_ids(first_visible_gpu())
    print('ALS backend:', ALS_BACKEND_DIR)
    print('Reconstruction settings:', RECON_SETTINGS)
    print('Scheduler GPU assignment:', os.environ.get('CUDA_VISIBLE_DEVICES', '(unset)'))
    configure_context(BACKEND_RECORD, RECON_SETTINGS)
    print('DONE: backend verified and context configured; estimate COR next.')
except Exception:
    print("FAILED: reconstruction-settings; see exception below.")
    raise

## Zoom into image details

Use **Zoom** to enlarge a region and **Center x / Center y** or the arrow buttons
to move around it. **Reset view** returns to the full image. Comparison panels
share the same pixel coordinates, so reconstructed, denoised and residual features
stay aligned. The coordinates and crop size are shown above the images.

**Black / White** adjust display contrast. Matching reconstruction panels share
one contrast window; the residual has its own window. Choose the window with the
Contrast menu. Windows stay fixed when zooming or changing slices; **Reset contrast**
restores the initial full-image window.

Displays use pixel-preserving crops and nearest-neighbor magnification. Zoom does
not change saved reconstructions or run any reconstruction/inference. TIFF browsers
load one row at a time and retain their zoom and contrast when the row changes.

Controls use the existing `ipywidgets` support. If widgets are unavailable, set
`IMAGE_VIEW_OPTIONS = {'zoom': 8, 'center': (x, y)}` in cell 3 and rerun the display
cell for a static crop. No interactive Matplotlib backend is required.

## Optional first/opposite projection check

Skip this section for the usual Vo workflow. The next cell provides a phase-correlation
estimate and an alignment overlay as an independent visual check. It does not supply
Vo's starting point or affect a saved plan.

In [ ]:
# 13. Check projection alignment (optional)

# USER INPUT: OPTIONAL brightness ratio (unitless).
try:
    BRIGHT_RATIO = 1.0
    captured_alignment = context_token()
    assert_context(captured_alignment, check_source=True)
    first_projection, last_projection, opposite_index = alignment_projections(RAW_H5, BRIGHT_RATIO)
    CENTER_ESTIMATE, displacement = center_estimate(first_projection, last_projection)
    print(f'Alignment COR: {CENTER_ESTIMATE:+.3f} detector pixels; opposite index {opposite_index}')
    alignment_figure(first_projection, last_projection, CENTER_ESTIMATE)
    center_control = None
    print('DONE: optional alignment preview; run native Vo COR to estimate and save the plan.')
except Exception:
    print("FAILED: center-overlay; see exception below.")
    raise

## Find COR with TomoPy Vo and save the run

Run the next cell, then **enqueue and reconstruct**. It reads three detector-row
sinograms through the same ALS flat/dark normalization, transmission floor, minus-log
and selected preprocessing/postprocessing used for reconstruction. It calls native
`tomopy.find_center_vo()` on each and saves their median midpoint offset when they agree.
No candidate images are reconstructed or scored to estimate COR.

The default rows are at 25%, 50%, and 75% of detector height; choose rows containing
sample if those are empty. Nonfinite/flat rows, estimates near the native Vo limits,
and a row spread greater than 1 pixel require review. The threshold is provisional,
not calibrated accuracy. Approximately uniform 180-degree angles are required.

`VO_OPTIONS` overrides the native parameters shown in `vo_options()`; the default
fine step is 0.1 pixels and `ncore=1` limits native parallelism on supported TomoPy
versions. A fine step is numerical resolution, not measured accuracy. Vo internally
uses its own Fourier coarse/fine procedure; this notebook does not use a reconstruction-image sharpness grid to find COR. Results and logs are saved
under `auto_cor_vo` with the TomoPy version and both detector-center/offset values.

If **REVIEW REQUIRED** appears, change the rows or native bounds, or apply a reviewed
manual COR below. Optional alignment and image previews are available for inspection.
An existing supported run plan restores its saved COR without re-estimation.
Use a **new run label** to obtain a new Vo estimate for a previously frozen scan.

In [ ]:
# 14. Estimate COR with TomoPy Vo

# USER INPUT: OPTIONAL detector rows; ADVANCED native Vo parameters.
try:
    AUTO_COR_ROWS = None  # None selects rows at 25%, 50%, and 75% of detector height.
    VO_OPTIONS = {}  # Defaults in vo_options(); e.g. {'ncore': 1, 'step': 0.1}.
    AUTO_COR_RESULT = None
    plan = None
    FINAL_CENTER = None
    captured_cor = context_token()
    PLAN_FILE = RUN_DIR/'run_plan.json'
    plan, AUTO_COR_RESULT = automatic_cor_workflow(captured_cor, INFO, AUTO_COR_ROWS, VO_OPTIONS)
    if AUTO_COR_RESULT is not None:
        for row, center, error in zip(AUTO_COR_RESULT['rows'], AUTO_COR_RESULT['row_centers'], AUTO_COR_RESULT['row_errors']):
            print(f'Row {row}: {center:+.3f} px' if error is None else f'Row {row}: {error}')
        print('Row spread:', AUTO_COR_RESULT['row_spread'], 'px | elapsed:',
              round(AUTO_COR_RESULT['elapsed_seconds'], 2), 'seconds')
        if not AUTO_COR_RESULT['ncore_supported']:
            print('This TomoPy release chooses its own worker count; ncore is unsupported.')
    if plan is not None:
        FINAL_CENTER = float(plan['reconstruction']['center'])
        print(f'DONE: COR {FINAL_CENTER:+.3f} px saved in {PLAN_FILE}. Enqueue this run next.')
        if AUTO_COR_RESULT is None:
            print('Restored existing plan; no COR estimation repeated.')
    else:
        print('REVIEW REQUIRED:', ', '.join(AUTO_COR_RESULT['diagnostics']))
        print('No plan saved. Choose informative rows or use optional manual review below.')
except Exception:
    print('FAILED: native Vo COR; see exception below.')
    raise

In [ ]:
# 15. Review COR manually (optional)

# USER INPUT: OPTIONAL manual review, rows and numeric fallback; offsets in detector pixels.
try:
    RUN_MANUAL_REFINEMENT = False
    MANUAL_ROWS = [NROWS//2]
    MANUAL_ADJUSTMENT_PIXELS = 0.0  # Adjustment relative to the current estimate
    APPLY_NUMERIC_MANUAL = False  # Explicit application, separate from control position.
    if not RUN_MANUAL_REFINEMENT:
        print('SKIPPED: manual refinement disabled; enqueue a saved plan, or enable review if needed.')
    else:
        captured_review = context_token()
        base, reason = manual_start(captured_review, globals().get('AUTO_COR_RESULT'), globals().get('CENTER_ESTIMATE'))
        print(f'Review starts at {base:+.3f} px: {reason}')
        if widgets:
            adjustment = widgets.FloatSlider(value=0., min=-1., max=1., step=.05,
                description='Adjust (px):', continuous_update=False, readout_format='.2f')
            rows_control = widgets.Text(value=','.join(map(str,MANUAL_ROWS)), description='Rows:')
            absolute = widgets.HTML(value=f'Absolute midpoint COR: {base:+.3f} px')
            preview_button = widgets.Button(description='Preview selected COR')
            apply_button = widgets.Button(description='Apply selected COR')
            review_output = widgets.Output()
            # Default arguments capture this widget generation, never mutable notebook globals.
            def update_absolute(change, captured=captured_review, origin=base, label=absolute):
                assert_context(captured)
                label.value = f'Adjustment {change["new"]:+.2f} px; absolute midpoint COR {origin+change["new"]:+.3f} px'
            def review_click(button, captured=captured_review, origin=base, slider=adjustment,
                             row_input=rows_control, output=review_output):
                with output:
                    assert_context(captured, check_source=True)
                    rows = sorted(set(int(v.strip()) for v in row_input.value.split(',')))
                    center = origin+slider.value
                    with exclusive_lock(Path(captured['run_dir'])/'.run.lock'):
                        cor_comparison(captured, [center], rows,
                                       Path(captured['run_dir'])/'manual_cor_previews')
                    print('DONE: selected-COR preview; apply the selected offset explicitly.')
            def apply_click(button, captured=captured_review, origin=base, slider=adjustment,
                            row_input=rows_control, output=review_output):
                with output:
                    rows = sorted(set(int(v.strip()) for v in row_input.value.split(',')))
                    apply_manual(captured, origin+slider.value, rows, 'manual_widget', origin)
                    print('DONE: manual COR applied; freeze the plan next.')
            adjustment.observe(update_absolute, names='value')
            preview_button.on_click(review_click); apply_button.on_click(apply_click)
            display(adjustment, absolute, rows_control, widgets.HBox([preview_button,apply_button]), review_output)
            print('READY: manual review controls; slider movement does not apply a selection.')
        else:
            center = base+MANUAL_ADJUSTMENT_PIXELS
            with exclusive_lock(RUN_DIR/'.run.lock'):
                cor_comparison(captured_review, [center], MANUAL_ROWS,
                               RUN_DIR/'manual_cor_previews')
            if APPLY_NUMERIC_MANUAL:
                apply_manual(captured_review, center, MANUAL_ROWS, 'manual_numeric', base)
                print('DONE: numeric manual selection applied; freeze next.')
            else:
                print('READY: numeric comparison; set APPLY_NUMERIC_MANUAL=True to apply it.')
    print('DONE: optional manual refinement action finished; see outcome above.')
except Exception:
    print("FAILED: run-preview; see exception below.")
    raise

## Optional: save a reviewed COR

Skip this cell when the automatic COR cell has already saved the plan. After manual review,
click **Apply selected COR**, then run the next cell. Alternatively, enter a numeric
midpoint offset in `COR_OVERRIDE` (zero and negative values are valid).

The cell also restores an existing plan after a restart. Frozen reconstruction settings
and COR cannot be changed under the same run label; use a new label for a different COR.
Moving a slider alone does not apply a selection.

In [ ]:
# 16. Save a reviewed COR (optional)

# USER INPUT: OPTIONAL numeric midpoint COR_OVERRIDE; run only for review or restore.
try:
    COR_OVERRIDE = None  # None uses the applied manual COR or saved plan.
    COR_POLICY = 'auto_if_eligible'
    captured_freeze = context_token()
    PLAN_FILE = RUN_DIR/'run_plan.json'
    existing = read_plan(PLAN_FILE) if PLAN_FILE.exists() else None
    FINAL_CENTER, selection = resolve_center(captured_freeze, globals().get('MANUAL_COR_SELECTION'),
        COR_OVERRIDE, existing, globals().get('AUTO_COR_RESULT'), COR_POLICY)
    plan = freeze_run(captured_freeze, INFO, FINAL_CENTER, selection)
    print('Frozen plan:', PLAN_FILE, '| midpoint COR:', FINAL_CENTER, 'px')
    print('DONE: schema-4 run plan frozen; enqueue this run next.')
except Exception:
    print("FAILED: freeze-plan; see exception below.")
    raise

## Queue and reconstruct: cells 17–18

Cell 17 adds the selected `run_plan.json` to
`OUTPUT_ROOT/reconstruction_queue_v3.json`. Run it only after COR has produced a
saved plan. Cell 18 prepares even/odd projection views and reconstructs queued
runs **sequentially**; adding an entry alone does not start reconstruction.

The queue can contain several scans/run labels. It records a stable `id` and a
`plan_path` for each entry. Preflight checks each plan and its source/settings;
missing or mismatched entries are blocked, duplicates skipped, and ordinary errors
recorded while later valid runs continue. An interrupt stops further entries.
The queue report is `OUTPUT_ROOT/queue_report_v3.json`; each run also has logs and
completion records. On a retry, completed stacks are skipped only after their
TIFFs pass validation; an incomplete tracked stage is reconstructed again.

Cell 19 inspects the run selected in cell 10, not every run that happened to be in
the queue. To remove an entry or repair its path, use cells 26–27. Removing a queue
entry does not delete reconstructed images. The queue lock is
`OUTPUT_ROOT/reconstruction_queue_v3.json.lock`; run actions can also use a
`.run.lock` inside their run directory. A lock file marks an active operation;
confirm the owner has stopped before considering stale-lock repair.


In [ ]:
# 17. Add the run to the reconstruction queue

# USER INPUT: OPTIONAL queue location under OUTPUT_ROOT.
try:
    QUEUE_FILE = OUTPUT_ROOT/'reconstruction_queue_v3.json'
    assert_context(context_token(), check_source=True)
    queue = enqueue_plan(QUEUE_FILE, PLAN_FILE)
    print('Queued entries:', len(queue['entries']))
    print('DONE: selected frozen run enqueued; execute queue next.')
except Exception:
    print("FAILED: enqueue; see exception below.")
    raise

In [ ]:
# 18. Reconstruct queued runs

# USER INPUT: REQUIRED verify backend/settings above; OPTIONAL queue path.
try:
    QUEUE_FILE = OUTPUT_ROOT/'reconstruction_queue_v3.json'
    require_compute_session()
    report = execute_queue(QUEUE_FILE, BACKEND_RECORD, RECON_SETTINGS)
    counts = {status:sum(e['status']==status for e in report['outcomes']) for status in
              ('completed','already-complete','skipped','blocked','failed')}
    print(('DONE WITH ERRORS' if counts['blocked'] or counts['failed'] else 'DONE')+': queue finished;', counts)
    print('DONE: queue outcomes saved in queue_report_v3.json; inspect selected-run stacks next.')
except Exception:
    print("FAILED: run-queue; see exception below.")
    raise

## Check the three reconstruction stacks: cell 19

The same row should show the same structures in all three images. Even and odd reconstructions will be noisier and can have more angular undersampling artifacts. Their measured projection indices are disjoint, although shared flat/dark calibration and detector artifacts can leave correlated errors; denoising is not a substitute for this check. Preserve float32 intensity values. The viewer uses the full reconstruction's contrast range for all three images.

Check more than the middle slice, including regions near the top and bottom. Stop before training if boundaries are displaced, the stacks have different orientation, or one view shows a gross intensity mismatch. All output filenames represent the same zero-based detector row: `recon_00000.tiff` is row 0. Shared contrast helps reveal differences but is not a quantitative test of independence or reconstruction accuracy.

In [ ]:
# 19. Check reconstruction stacks

# USER INPUT: OPTIONAL row browser; selected run is PLAN_FILE, independent of queue order.
try:
    PAIR_CONTEXT = context_token()
    assert_context(PAIR_CONTEXT, check_source=True)
    active_plan = read_plan(PLAN_FILE)
    resolve_center(PAIR_CONTEXT, existing=active_plan)
    FULL_DIR = RUN_DIR / active_plan['reconstruction']['outputs']['full']
    EVEN_DIR = RUN_DIR / active_plan['reconstruction']['outputs']['even']
    ODD_DIR = RUN_DIR / active_plan['reconstruction']['outputs']['odd']
    stack_lists = [validate_stack(p, NROWS, NCOLS) for p in (FULL_DIR, EVEN_DIR, ODD_DIR)]
    def load_pair_images(row, captured=PAIR_CONTEXT, paths_by_stage=tuple(stack_lists)):
        """Load only the selected row; shared crop coordinates keep comparisons aligned."""
        assert_context(captured)
        images = [tifffile.imread(paths[row]) for paths in paths_by_stage]
        if not all(np.isfinite(image).all() for image in images):
            raise ValueError('Nonfinite reconstruction values.')
        return images
    PAIR_VIEWER = zoom_image_viewer(load_pair_images(NROWS//2),
        titles=['Full','Even projections','Odd projections'],captured=PAIR_CONTEXT,
        title='Reconstruction pairs',load_images=load_pair_images,count=NROWS,index=NROWS//2,index_label='Row')
    print('READY: selected-run paired-stack check; inspect structures before training.')
except Exception:
    print("FAILED: pair-qc; see exception below.")
    raise

## Configure and train

Fresh 2.5D runs default to `nafnet25d`; set `TRAIN_ARCHITECTURE = 'unet'` in
cell 21 for the baseline. **2.5D** uses neighboring axial slices as input and
predicts their center slice. **3D** uses cubic patches and predicts a volume patch;
it uses the 3D U-Net and generally needs more computation/memory. Cell 20 selects
the mode, and cell 21 applies its preset. Compare five slices first,
then change `n_slices` to 7 or 9 in a fresh run. BF16 is optional; set FP32 in the
preset for a precision comparison. These are starting parameters, not tuned optima.

Cell 22 prints a disabled message until `RUN_TRAINING = True`.
It then uses all allocated GPUs, retains the live log box, and saves automatic
history plots even on interruption. Patience uses raw evaluation loss, and the
10-epoch history smoothing affects only the plot.

| Setting in cell 21's preset/overrides | Meaning |
|---|---|
| `psz` / `n_slices` | 2.5D square patch width and number of neighboring slices; use an odd slice count |
| `mbsz` | Batch size per GPU; four GPUs with batch 4 process a total batch of 16 |
| `precision` | `bf16` for supported GPUs such as A100; `fp32` for a precision comparison |
| `psz_3d` / `nb_patches_3d` | 3D cube width and number of sampled training patches per epoch |
| `lr` | Adam learning rate; YAML prepared for a new run starts at 0.001 |
| `warmup` | Optimizer updates before adding weighted LCL; initial YAML uses 2000 |
| `maxep` / `patience` | Maximum epochs / consecutive eligible epochs without lower evaluation L1; defaults 2000 / 100 |

An **epoch** visits the sampled training patches and then runs an evaluation pass.
Evaluation currently samples the training dataset again; it is not a separate
held-out accuracy estimate. L1 measures paired absolute error. LCL is an output
contrast regularizer weighted by 0.01 after warmup. L2/MSE and SSIM loss are
recorded only for reference. No learning-rate scheduler is enabled.

Saved training settings and normalization are kept when `TrainOutput` exists;
editing a fresh-run preset does not alter an existing model. Cell 22 automatically
adds `--resume` when a valid `resume.pth` exists, otherwise starts fresh. A U-Net
checkpoint cannot initialize NAFNet. Run cell 23 on a saved log while
leaving training disabled.


In [ ]:
# 20. Choose the training mode

# USER INPUT: REQUIRED training mode.
try:
    # Select before running the configuration cell below. Use a new RUN_LABEL for a new mode.
    TRAIN_MODE = '2.5d'  # '2.5d' or '3d'; also the fallback when widgets are unavailable
    if widgets:
        mode_picker = widgets.Dropdown(
            options=[('2.5D — neighboring slices', '2.5d'), ('3D — cubic patches', '3d')],
            value=TRAIN_MODE, description='Train mode:', layout=widgets.Layout(width='450px'))
        display(mode_picker)
    else:
        mode_picker = None
        print('Training mode:', TRAIN_MODE)
    print('READY: training mode selector; configure training next.')
except Exception:
    print("FAILED: training-mode-selector; see exception below.")
    raise

In [ ]:
# 21. Configure training

# USER INPUT: ADVANCED per-mode presets and TRAIN_OVERRIDES below.
try:
    # Choose the mode above, then run this cell. Settings below apply only to fresh training.
    SELECTED_MODE = mode_picker.value if mode_picker is not None else TRAIN_MODE
    if SELECTED_MODE not in ('2.5d', '3d'):
        raise ValueError("TRAIN_MODE must be '2.5d' or '3d'.")
    TRAIN_GPU_IDS = all_allocated_gpus()
    TRAIN_ARCHITECTURE = 'nafnet25d'  # 'nafnet25d' or 'unet' for 2.5D; 3D uses unet3d
    TRAIN_PRESETS = {
        '2.5d': {'psz': 256, 'n_slices': 5, 'mbsz': 4, 'architecture': TRAIN_ARCHITECTURE,
                 'precision': 'bf16' if TRAIN_ARCHITECTURE == 'nafnet25d' else 'fp32', 'num_workers': 0},
        '3d': {'psz': 256, 'n_slices': 5, 'psz_3d': 96, 'nb_patches_3d': 17600,
               'n_blocks_3d': 4, 'start_filts_3d': 56, 'mbsz': 1, 'architecture': 'unet3d',
               'precision': 'fp32', 'num_workers': 0},
    }
    TRAIN_OVERRIDES = dict(TRAIN_PRESETS[SELECTED_MODE], mode=SELECTED_MODE,
                           z_stride=1, maxep=2000, patience=100)
    
    
    
    
    assert_context(PAIR_CONTEXT, check_source=True)
    CONFIG_YAML = RUN_DIR / (FULL_DIR.name + '_config.yaml')
    TRAIN_DIR = RUN_DIR / 'TrainOutput'
    with exclusive_lock(RUN_DIR/'.run.lock'):
        if TRAIN_DIR.exists() and not CONFIG_YAML.is_file():
            raise FileNotFoundError('TrainOutput exists but its YAML is missing. Restore the original training YAML; do not recreate it.')
        run_logged(DENOISE_CMD + ['train', '-h'], RUN_DIR / 'logs/denoise_help.log', RUN_DIR)
        help_text = (RUN_DIR / 'logs/denoise_help.log').read_text(encoding='utf-8')
        assert all(flag in help_text for flag in ('--mode', '--resume', '--no-search', '--architecture')), \
            'Noise2Inverse CLI differs from the reviewed version; inspect installation before training.'
        if not CONFIG_YAML.exists():
            run_logged(DENOISE_CMD + ['prepare', '--file-name', str(RAW_H5), '--out-path-name', str(FULL_DIR)],
                       RUN_DIR / 'logs/prepare.log', RUN_DIR)
        cfg = yaml.safe_load(CONFIG_YAML.read_text(encoding='utf-8'))
        if not isinstance(cfg, dict) or not isinstance(cfg.get('train'), dict):
            raise ValueError('Training YAML is malformed; restore it before continuing.')
        if TRAIN_DIR.exists() and cfg['train'].get('mode', '2.5d') != SELECTED_MODE:
            raise ValueError('Selected mode differs from the saved model. Select its original mode to resume, or choose a new RUN_LABEL for fresh training.')
        if TRAIN_DIR.exists() and cfg['train'].get('architecture',
                'unet3d' if SELECTED_MODE == '3d' else 'unet') != TRAIN_OVERRIDES['architecture']:
            raise ValueError('Saved architecture differs. Use its original architecture or a new run label.')
        if not TRAIN_DIR.exists():
            cfg['train'].update(TRAIN_OVERRIDES)
            cfg['infer'].update(precision=cfg['train']['precision'], num_workers=0, max_batch_size=32)
        validate_denoise_config(cfg, RUN_DIR, FULL_DIR, EVEN_DIR, ODD_DIR, NCOLS, nrows=NROWS)
        if not TRAIN_DIR.exists():
            temporary = CONFIG_YAML.with_suffix('.yaml.tmp')
            temporary.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
            temporary.replace(CONFIG_YAML)
        else:
            print('Existing training directory: keeping saved training settings and normalization.')
        print(yaml.safe_dump(cfg, sort_keys=False))
        print("Ignore prepare's printed split commands: the paired stacks are already verified.")
        print('Training mode:', cfg['train'].get('mode', '2.5d'))
        print('Training GPUs:', TRAIN_GPU_IDS)
        print('Batch per GPU:', cfg['train']['mbsz'], '| total batch:', cfg['train']['mbsz'] * len(TRAIN_GPU_IDS.split(',')))
        print('Paired input arrays across training processes:', round(2 * NROWS * NCOLS**2 * 4 * len(TRAIN_GPU_IDS.split(',')) / 2**30, 1), 'GiB, plus loading/normalization/worker overhead.')
        TRAINING_CONTEXT = context_token()
    print('DONE: selected-run training configured; train next.')
except Exception:
    print("FAILED: configure-training; see exception below.")
    raise

In [ ]:
# 22. Train or resume the denoising model

RUN_TRAINING = False  # Enable training or resume explicitly
if not RUN_TRAINING:
    print('Training disabled. Set RUN_TRAINING=True in this cell when ready.')
else:

    # USER INPUT: OPTIONAL resume is determined by saved training outputs; uses all assigned GPUs.
    try:
        assert_context(TRAINING_CONTEXT, check_source=True)
        require_compute_session()
        TRAIN_GPU_IDS = all_allocated_gpus()
        cfg = yaml.safe_load(CONFIG_YAML.read_text(encoding='utf-8'))
        validate_denoise_config(cfg, RUN_DIR, FULL_DIR, EVEN_DIR, ODD_DIR, NCOLS, nrows=NROWS)
        if cfg['train'].get('mode', '2.5d') != SELECTED_MODE or (mode_picker is not None and mode_picker.value != SELECTED_MODE):
            raise ValueError('Mode selection changed. Rerun the cell 21 configuration cell before training.')
        if cfg['train'].get('mode') == '3d' and cfg['train']['nb_patches_3d'] < len(TRAIN_GPU_IDS.split(',')):
            raise ValueError('nb_patches_3d must provide at least one sample per GPU.')
        print('Using all allocated training GPUs:', TRAIN_GPU_IDS)
        for folder in (FULL_DIR, EVEN_DIR, ODD_DIR):
            validate_stack(folder, NROWS, NCOLS)
        resume = TRAIN_DIR.exists()
        if resume:
            assert (TRAIN_DIR / 'resume.pth').is_file(), \
                'TrainOutput exists without a resume checkpoint. Preserve it and use a new run for fresh training.'
        cmd = DENOISE_CMD + ['train', '--config', str(CONFIG_YAML), '--gpus', TRAIN_GPU_IDS,
                             '--mode', cfg['train'].get('mode', '2.5d'), '--no-search']
        if resume:
            cmd.append('--resume')
        training_run_dir = Path(TRAINING_CONTEXT['run_dir'])
        training_log = training_run_dir/'logs/train.log'
        TRAINING_HISTORY = None
        with exclusive_lock(training_run_dir/'.run.lock'):
            training_outcome = 'completed'
            try:
                run_logged(cmd, training_log, training_run_dir)
            except BaseException:
                training_outcome = 'interrupted or failed'
                raise
            finally:
                try:
                    TRAINING_HISTORY = plot_training_history(
                        training_log, training_run_dir/'training_history', show=True, outcome=training_outcome)
                except Exception as plot_error:
                    print('Training history could not be plotted:', plot_error)
                    print('Rerun the history cell after loading the log; the training outcome is preserved.')
        print('DONE: training command finished; training history saved when available; inspect denoising next.')
    except Exception:
        print("FAILED: train; see exception below.")
        raise

## Training history

New runs also log L2/MSE and SSIM loss for reference. The automatic plot adds
reference panels when present; the CSV includes them and the fixed SSIM intensity
range. They do not affect optimization, checkpoints or patience. Older logs remain supported.

Training automatically saves and displays the raw epoch-average L1/LCL losses with
a trailing 10-epoch moving average, warmup marker, and best-checkpoint annotations.
The run's `training_history` folder contains PNG, SVG, and extracted CSV files.
It also attempts a partial plot after interruption or failure, when complete epochs exist.

Run the next cell to regenerate a plot for an existing run, or to refresh it from
a growing log. Set `HISTORY_LOG` to another training log if needed. The history cell reads text; it does not load a checkpoint, train a model, or require PyTorch in the ALS kernel. Resumed epochs are
combined, with the latest complete record used when an epoch appears twice.

In a line such as `L1 loss: mean, first => last`, `mean` is the epoch's average
batch value, followed by its first and last batch values. `rate` is seconds for
the complete training/evaluation epoch. L1 and weighted LCL logs sum both paired
directions; new L2/MSE and SSIM-loss logs average them. They report the final GPU
rank's batches, not an aggregate across GPUs.

SSIM loss is `1 - SSIM`; smaller means greater paired similarity. It uses a fixed
normalized-intensity range in `train.ssim_data_range`, inferred once from the two
training stacks if absent and saved in YAML. Compare SSIM only with consistent
range/settings. In 3D, its SSIM/LCL diagnostics use the center axial plane; MSE
uses the entire patch. Metrics against noisy paired targets are not clean-image
accuracy measurements. Reference CSV is also written to
`TrainOutput/results/reference_metrics.csv` and retained on resume.

The live log box shows a scrolling tail while a command runs; the complete log is
saved under the printed `logs/` path. Startup messages may repeat once per GPU
process. ANSI color escapes are stripped from the displayed text. Refreshing
cell 23 reads the current log once; rerun it for another refresh.


In [ ]:
# 23. Show or regenerate training history (optional)

# USER INPUT: OPTIONAL external log path; None uses the selected run's logs/train.log.
try:
    HISTORY_LOG = None  # Example: Path('/pscratch/.../logs/train.log')
    MOVING_AVERAGE_EPOCHS = 10
    if HISTORY_LOG is None:
        history_context = context_token()
        assert_context(history_context)
        history_run_dir = Path(history_context['run_dir'])
        history_log = history_run_dir/'logs/train.log'
        history_output = history_run_dir/'training_history'
    else:
        history_log = Path(HISTORY_LOG).expanduser().resolve()
        history_output = history_log.parent/'training_history'
    TRAINING_HISTORY = plot_training_history(history_log, history_output,
        moving_window=MOVING_AVERAGE_EPOCHS, show=True)
    print('DONE: training history action finished; see plot or availability message above.')
except Exception:
    print('FAILED: show training history; see exception below.')
    raise

## Inspect denoising: cell 24

2.5D generates a single-slice preview. In 3D, the upstream slice command is unavailable; this cell selects the checkpoint and prints next steps until cell 25 has produced a volume, then displays the selected row and residual from that existing volume. Review `TrainOutput/results` before the first 3D volume run. Changing the checkpoint after a volume exists does not regenerate that volume.

Choose an available checkpoint (`lcl`, `val`, or `edge`) and compare the original, denoised image, and residual at the same row. Look for loss of pores, interfaces, or small structures. The residual should be checked for real features, not judged by smoothness alone.

| `CHECKPOINT` | Checkpoint selection criterion |
|---|---|
| `lcl` | Lowest recorded validation LCL loss. |
| `val` | Lowest recorded validation L1 loss. |
| `edge` | Highest recorded edge score. |

These are internal model-selection scores, not comparisons with a clean reference. A larger edge score can retain noise as well as detail. Compare several rows for removed pores, softened interfaces, and changes in fine structure. Visible anatomy in the residual can indicate removed signal. 

`TEST_SLICE` is the zero-based row index in the full TIFF stack. Rerunning the same test slice writes the same output filename, including when changing checkpoints; preserve a result under another name before comparing saved files later. Cell 25 uses the `CHECKPOINT` selected in cell 24 and the normalization saved in the YAML.

In [ ]:
# 24. Test denoising on one slice

# USER INPUT: OPTIONAL checkpoint/test row and inference GPU.
try:
    CHECKPOINT = 'lcl'  # Try 'val' or 'edge' if available; compare preserved detail
    TEST_SLICE = NROWS // 2
    assert_context(TRAINING_CONTEXT, check_source=True)
    require_compute_session()
    INFERENCE_GPU_ID = validate_gpu_ids(INFERENCE_GPU_ID)
    validate_checkpoint(TRAIN_DIR, CHECKPOINT)
    cfg = yaml.safe_load(CONFIG_YAML.read_text(encoding='utf-8'))
    validate_denoise_config(cfg, RUN_DIR, FULL_DIR, EVEN_DIR, ODD_DIR, NCOLS, nrows=NROWS, normalized=True)
    assert 0 <= TEST_SLICE < NROWS
    mode = cfg['train'].get('mode', '2.5d')
    base_name = FULL_DIR.name[:-4] if FULL_DIR.name.endswith('_rec') else FULL_DIR.name
    if mode == '2.5d':
        with exclusive_lock(RUN_DIR/'.run.lock'):
            run_logged(DENOISE_CMD + ['slice', '--config', str(CONFIG_YAML), '--slice-number', str(TEST_SLICE),
                                     '--gpus', INFERENCE_GPU_ID, '--mode', mode, '--checkpoint', CHECKPOINT],
                       RUN_DIR / 'logs/slice.log', RUN_DIR)
        denoised_file = RUN_DIR / (base_name + '_denoised_slices' + ('_nafnet25d' if cfg['train'].get('architecture') == 'nafnet25d' else '')) / f'{TEST_SLICE:05d}.tiff'
    else:
        denoised_file = RUN_DIR / (base_name + '_denoised_volume_3d') / f'{TEST_SLICE:05d}.tiff'
        if not denoised_file.is_file():
            print('3D has no upstream single-slice preview. Review training previews in TrainOutput/results, run cell 25, then rerun this cell to inspect this row and its residual.')
        else:
            print('Inspecting the existing 3D volume; changing CHECKPOINT here does not regenerate it.')
    if denoised_file.is_file():
        before = tifffile.imread(stack_lists[0][TEST_SLICE])
        after = tifffile.imread(denoised_file)
        if after.shape != before.shape or not np.isfinite(after).all() or not np.isfinite(before).all():
            raise ValueError('Original and denoised images must have matching shapes and finite pixels.')
        lo, hi = image_display_limits(before)
        residual = before - after
        span = max(float(np.percentile(np.abs(residual), 99)), 1e-8)
        DENOISING_VIEWER = zoom_image_viewer([before,after,residual],
            titles=['Reconstructed','Denoised','Residual: reconstructed − denoised'],
            limits=[(lo,hi),(lo,hi),(-span,span)],cmaps=['gray','gray','coolwarm'],
            captured=TRAINING_CONTEXT,title=f'Denoising inspection — row {TEST_SLICE}')
    print('DONE: denoising inspection action finished; review before volume inference.')
except Exception:
    print("FAILED: single-slice; see exception below.")
    raise

## Denoise the full volume: cell 25

For 2.5D, review several test slices in cell 24 first. For 3D, review the training
preview images, run volume inference, then use cell 24 to inspect the saved rows.
Select an available checkpoint before starting. Training can use several GPUs;
this volume command uses **one GPU**, plus substantial **host RAM**.

This repository's 2.5D profiler uses the actual input channel count; models with
5, 7 or 9 slices are supported. Its prediction buffer is **float32**. For a
624 × 2560 × 2560 volume with 256-pixel patches and 50% overlap, that buffer alone
is about **55 GiB**: `624 × 19 × 19 × 256 × 256 × 4 bytes`. Input, stitching,
temporary arrays and the notebook process require additional RAM. This is a
planning estimate, not a measured peak, and GPU VRAM cannot replace host RAM.
3D stitches predictions incrementally but retains the full input and accumulation
arrays; it still needs substantial host RAM.

Neither the notebook nor this backend deletes an existing volume directory to
restart. Cell 25 refuses to overwrite it. There is **no volume resume**: preserve
partial output and use a new output directory/run for a retry. The CLI's
`--output-dir` can select a separate destination outside this cell, but assembling
independently inferred subvolumes is not automated here. If full 2.5D inference
does not fit, its single-slice command remains available.

NAFNet outputs end in `_nafnet25d`, so they are distinguishable from baseline
outputs. After inference, the notebook checks filenames, dimensions, floating
dtype and finite pixels one slice at a time, then opens a zoomable browser. These
checks add an output-volume read and establish file consistency. Inspect fine
features and residuals before accepting the result.


In [ ]:
# 25. Denoise the full volume

# USER INPUT: OPTIONAL checkpoint/test row and inference GPU.
try:
    assert_context(TRAINING_CONTEXT, check_source=True)
    require_compute_session()
    INFERENCE_GPU_ID = validate_gpu_ids(INFERENCE_GPU_ID)
    validate_checkpoint(TRAIN_DIR, CHECKPOINT)
    cfg = yaml.safe_load(CONFIG_YAML.read_text(encoding='utf-8'))
    validate_denoise_config(cfg, RUN_DIR, FULL_DIR, EVEN_DIR, ODD_DIR, NCOLS, nrows=NROWS, normalized=True, volume=True)
    mode = cfg['train'].get('mode', '2.5d')
    base_name = FULL_DIR.name[:-4] if FULL_DIR.name.endswith('_rec') else FULL_DIR.name
    VOLUME_DIR = RUN_DIR / (base_name + '_denoised_volume_' + mode + ('_nafnet25d' if cfg['train'].get('architecture') == 'nafnet25d' else ''))
    assert not VOLUME_DIR.exists(), f'Volume output already exists: {VOLUME_DIR}. Preserve it before restarting.'
    with exclusive_lock(RUN_DIR/'.run.lock'):
        run_logged(DENOISE_CMD + ['volume', '--config', str(CONFIG_YAML), '--gpus', INFERENCE_GPU_ID,
                                 '--mode', mode, '--checkpoint', CHECKPOINT],
                   RUN_DIR / 'logs/volume.log', RUN_DIR)
    output_files = validate_denoised_volume(VOLUME_DIR, NROWS, NCOLS)
    browse_tiffs(output_files, 'Denoised volume')
    print('DONE: volume inference validated; inspect output slices.')
except Exception:
    print("FAILED: volume; see exception below.")
    raise

## Optional queue inspection and repair: cells 26–27

Run cell 26 with its default empty selections to inspect the queue. The printed
`proposed.entries` contains each entry's `id` and `plan_path`; `preflight` reports
why an entry is valid, blocked or duplicated.

To remove one entry, copy its exact `id` into `REMOVE_ENTRY_IDS`, for example
`REMOVE_ENTRY_IDS = ['copied_entry_id']`, and rerun cell 26. To correct a moved
plan, put `{copied_entry_id: '/absolute/path/to/run_plan.json'}` into
`REPLACE_PLAN_PATHS`. Review the newly printed proposed entries, then set
`APPLY_QUEUE_REPAIR=True` in cell 27 and run it. Repair backs up the exact previous
queue bytes. Entry removal affects the queue only, not run files.

For corrupt queue JSON, set `REBUILD_FROM_PLANS` to an explicit list of valid
absolute plan paths in cell 26. Deduplication and removing confirmed missing
plans are separate opt-in settings; inspect their effects before applying.

If a lock is stale, first confirm its recorded job/kernel has stopped. Inspect
the lock's JSON, copy its token to `STALE_LOCK_TOKEN`, and use the confirmation
flags in cell 27. Never remove the lock of an active reconstruction.


In [ ]:
# 26. Inspect and prepare queue repair (optional)

# USER INPUT: OPTIONAL stable IDs/path replacements; defaults inspect without changes.
try:
    QUEUE_FILE = OUTPUT_ROOT/'reconstruction_queue_v3.json'
    REMOVE_ENTRY_IDS = []
    REPLACE_PLAN_PATHS = {}  # {stable_entry_id: absolute_schema4_plan_path}
    DEDUPLICATE_QUEUE = False
    REMOVE_CONFIRMED_MISSING = False
    REBUILD_FROM_PLANS = None  # Explicit list of schema-4 plans for corrupt/unsupported queue.
    QUEUE_REPAIR_PREVIEW = propose_queue_edit(QUEUE_FILE, REMOVE_ENTRY_IDS, REPLACE_PLAN_PATHS,
        DEDUPLICATE_QUEUE, REMOVE_CONFIRMED_MISSING, REBUILD_FROM_PLANS)
    print(json.dumps(QUEUE_REPAIR_PREVIEW, indent=2))
    print('READY: queue repair preview; review proposed entries before applying.')
except Exception:
    print("FAILED: queue-repair; see exception below.")
    raise

In [ ]:
# 27. Apply queue repair (optional)

# USER INPUT: OPTIONAL explicit repair opt-in; use the preceding preview.
try:
    APPLY_QUEUE_REPAIR = False
    REMOVE_STALE_QUEUE_LOCK = False
    ORIGINAL_JOB_OR_KERNEL_STOPPED = False
    STALE_LOCK_TOKEN = None  # Copy token from the inspected QUEUE_FILE.lock.
    if REMOVE_STALE_QUEUE_LOCK:
        remove_confirmed_stale_lock(queue_lock(QUEUE_FILE), ORIGINAL_JOB_OR_KERNEL_STOPPED, STALE_LOCK_TOKEN)
    if APPLY_QUEUE_REPAIR:
        backup = apply_queue_edit(QUEUE_FILE, QUEUE_REPAIR_PREVIEW)
        print('DONE: queue repair saved; backup:', backup)
    else:
        print('SKIPPED: queue repair application disabled.')
    print('DONE: optional queue repair action finished.')
except Exception:
    print("FAILED: queue-repair-apply; see exception below.")
    raise

## Optional scratch cleanup: cells 28–29

Use after stopping jobs that use the selected run. Cell 28 is a preview: select
categories in `CLEANUP_CATEGORIES`, rerun it, and inspect every proposed path,
file count, estimated bytes and stated effect. An empty list selects nothing.

| Category | Effect to consider |
|---|---|
| `full`, `even`, `odd` | Removes the corresponding reconstruction stack and invalidates its completion record; queue reconstruction can rebuild it |
| `projection_views` | Removes generated HDF5 view files, which point at the raw scan |
| `denoised` | Removes generated denoising outputs |
| `cor` | Selects generated COR review/preview paths listed in the preview |
| `logs` | Removes run logs; history cannot be regenerated from a deleted log |
| `training` | Removes model/checkpoint outputs and the scan training YAML, including saved normalization; resume/inference then require restoration or fresh training |

For deletion, copy the **exact resolved run directory** from the preview into
`CLEANUP_CONFIRM_RUN` in cell 29, set `CLEANUP_EXECUTE=True` and
`CLEANUP_DRY_RUN=False`, then run that cell. Defaults only preview. Rerun cell 28
after changing any selection. Raw scans, source code, environments, active locks
and symlink/junction paths are protected. Plans and queue entries are retained.

Preserve accepted raw data, TIFFs, model, YAML and plan in durable
storage before choosing cleanup. The report records failures and estimated
reclaimed bytes; it is not a storage backup and free-space updates may lag.


In [ ]:
# 28. Preview scratch cleanup (optional)

# USER INPUT: OPTIONAL generated categories; empty selection is the default.
try:
    CLEANUP_CATEGORIES = []  # cor, projection_views, full, even, odd, denoised, logs, training
    assert_context(context_token(), check_source=True)
    cleanup_plan = read_plan(PLAN_FILE)
    CLEANUP_PREVIEW = cleanup_manifest(cleanup_plan, OUTPUT_ROOT, CLEANUP_CATEGORIES)
    print(json.dumps(CLEANUP_PREVIEW, indent=2))
    print('READY: cleanup dry-run preview; no files deleted.')
except Exception:
    print("FAILED: scratch-cleanup-preview; see exception below.")
    raise

In [ ]:
# 29. Delete selected scratch outputs (optional)

# USER INPUT: OPTIONAL deletion opt-in and exact resolved run path confirmation.
try:
    CLEANUP_DRY_RUN = True
    CLEANUP_EXECUTE = False
    CLEANUP_CONFIRM_RUN = ''
    cleanup_report = execute_cleanup(cleanup_plan, OUTPUT_ROOT, CLEANUP_PREVIEW,
        CLEANUP_DRY_RUN, CLEANUP_CONFIRM_RUN, CLEANUP_EXECUTE)
    print('Cleanup:', cleanup_report['status'], '| estimated reclaimed bytes:', cleanup_report['estimated_reclaimed_bytes'])
    for failure in cleanup_report['failed']:
        print('DELETE FAILED:', failure)
    print('DONE: cleanup action finished; see dry-run or deletion report above.')
except Exception:
    print("FAILED: scratch-cleanup; see exception below.")
    raise

## Decisions and recovery

| Situation | What to rerun or inspect |
|---|---|
| Kernel restarted | Re-select ALS kernel, run 1–9, then 10–12 and 14 with the same scan/run settings; follow the restart sequence above |
| `Stale controls/results` | Rerun scan inspection, settings and the action that created that viewer; do not reuse controls from a different scan/run |
| COR says `REVIEW REQUIRED` | Choose informative detector rows/native Vo limits in 14, or review/apply a manual offset in 15 and save it in 16 |
| Source/backend/settings differ from a saved plan | Restore the original inputs/settings, or use a new run label; do not edit the frozen plan to force a match |
| Reconstruction interrupted | Inspect the run log and queue report, then rerun 17–18; validated completed stages are reused |
| Training interrupted | Preserve YAML/models/logs, rerun 19–21 with the original model settings, then enable 22 to resume from `resume.pth` |
| `TrainOutput` has no resume checkpoint | Preserve the folder; use a new run label for fresh training or restore its checkpoint |
| Volume inference interrupted | Preserve the partial output; no automatic volume resume exists. Retry with a new destination/run after resolving the error |
| Queue contains an unwanted/broken entry | Inspect and explicitly repair it in 26–27 |
| Reconstruction was cleaned up | Keep its plan, rerun 17–19 to rebuild/check stacks, then 20–21 before training |

Read the full log path printed by a failing cell; the live box shows only a tail.
GPU utilization alone does not prove an epoch has stalled. Check the process,
host memory and allocation time in the compute terminal. Never remove a lock or
delete outputs while its job/kernel is active.

Check denoising residuals for erased features and compare several rows at matched
zoom/contrast. Vo row agreement, low loss and a smooth image do not establish
scientific accuracy.


## Change notes and validation limits, v3.3

L2/MSE and SSIM loss are logged and saved for reference without changing the
L1/LCL objective. SSIM uses a fixed normalized intensity span; 3D SSIM evaluates
the center axial plane.

NAFNet is integrated in training, slice inference and volume inference, with
shared checkpoint metadata checks. Seven/nine-slice volume profiling uses the
actual input channel count. Border blending preserves intensities, and training
crops are taken before copying neighbor planes. BF16 and batch progress logging
are optional. GPU rank binding precedes NCCL initialization.

The notebook preserves v3.2 reconstruction geometry and plan/queue formats,
native Vo centering, linked zoom, live logs, and history plots. It was checked for
Python 3.8 syntax and configuration/output contracts. Backend shape/checkpoint and
synthetic TIFF inference tests are included; training tests are skipped by default.
The full reconstruction pipeline and multi-GPU NERSC training have not been run
as part of this integration. Scan quality and tiny-feature retention are unmeasured.
